# Phase 4 · Court judgment prediction

Predict the binary outcome of each case. Compare ending-only, beginning-and-ending, and hierarchical representations using development macro-F1. Train, validate, and export each experiment in order.

## 1. Required files and environment

- Attach `phase3_model_assets/` with the four Phase 1 checkpoints, tokenizers, and run configs.
- Attach `Phase_4_CJPE/` with `Train/train.json`, `Train/dev.json`, and `Test/test.json`. To download them in Kaggle, enable Internet and set the `HF_TOKEN` secret. The course evaluator is included.
- Enable a GPU for training. Use `MODEL_DIRS` or `DATA_ROOT` to resolve ambiguous inputs. Set `PREPARED_CACHE` to reuse token arrays and `RESUME_ROOT` to restore saved runs.

In [3]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'peft==0.17.1', 'transformers==4.55.4',
                'huggingface-hub>=0.34,<1', 'scikit-learn>=1.4,<2',
                'ijson>=3.3,<4', 'tqdm>=4.66,<5'], check=True)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.2/150.2 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 52.2 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'peft==0.17.1', 'transformers==4.55.4', 'huggingface-hub>=0.34,<1', 'scikit-learn>=1.4,<2', 'ijson>=3.3,<4', 'tqdm>=4.66,<5'], returncode=0)

In [4]:
import copy
import gc
import hashlib
import heapq
import itertools
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import time
import zipfile
from collections import Counter, defaultdict
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import HTML, display
import base64
import html
from sklearn.metrics import f1_score, precision_score, recall_score
from torch.utils.checkpoint import checkpoint
from torch.utils.data import DataLoader, Dataset

PAD, UNK, BOS, EOS = range(4)

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(path)

def digest_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def normalize_id(value):
    value = str(value)
    return value[:-4] if value.endswith(".txt") else value


import contextlib
import importlib.metadata
import ijson
from tqdm.auto import tqdm
from sklearn.metrics import confusion_matrix, classification_report
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from IPython.display import FileLink

VERSION = 'phase4_cjpe_v1'
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name}, {p.total_memory / 2**30:.2f} GiB')


PyTorch: 2.11.0+cu128 | CUDA: True
GPU 0: Tesla T4, 14.56 GiB
GPU 1: Tesla T4, 14.56 GiB


In [5]:
INPUT_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working/phase4_cjpe')
ASSET_DIR = OUTPUT_ROOT / 'assets'
DATA_ROOT = None
PREPARED_CACHE = None
RESUME_ROOT = None
MODEL_NAMES = ['model1', 'model2', 'model3', 'model4']
MODEL_DIRS = dict.fromkeys(MODEL_NAMES)
REPO = 'Exploration-Lab/CS-781-Capstone'
DATA_REVISION = None
LABEL_FIELD = None

DEFAULT = dict(
    seed=42, chunk_tokens=1024, overlap=128, max_chunks=6,
    representation='tail', aggregator='attention', head_dim=128,
    dropout=0.2, epochs=8, patience=3, effective_batch=8,
    micro_batch_per_gpu=1, activation_checkpointing=True,
    backbone_lr=2e-5, adapter_lr=1e-4, head_lr=2e-4,
    weight_decay=0.01, warmup_fraction=0.06, min_lr_ratio=0.1,
    clip=1.0, label_smoothing=0.0, class_weighting=False,
    rank=8, initial_rank=12, alpha=16, orth_reg_weight=0.5,
    model2_pool='encoder_query', save_every=100, log_every=20,
    max_hours=10.5, capacity_headroom_gib=1.5,
    kd_alpha=0.5, kd_temperature=2.0, student_keep=None,
)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ASSET_DIR.mkdir(parents=True, exist_ok=True)
if RESUME_ROOT is not None:
    previous = Path(RESUME_ROOT)
    for subfolder in ('runs','teacher_cache'):
        source = previous / subfolder
        destination = OUTPUT_ROOT / subfolder
        if source.is_dir():
            if destination.exists():
                raise FileExistsError(f'{destination} already exists. Set RESUME_ROOT=None after the first copy.')
            shutil.copytree(source,destination)
print('Output:', OUTPUT_ROOT)


Output: /kaggle/working/phase4_cjpe


## 2. Resolve checkpoint and dataset paths

In [6]:
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
    del token
except Exception:
    pass

def classify_model_config(config):
    if 'encoder_layers' in config and 'decoder_layers' in config:
        return 'model2'
    if 'kernel_size' in config and 'dilation_cycle' in config:
        return 'model4'
    if 'window' in config and 'global_every' in config:
        return 'model3'
    if 'layers' in config and 'heads' in config and 'context' in config:
        return 'model1'
    return None

def resolve_inputs():
    candidates = defaultdict(list)
    for path in INPUT_ROOT.rglob('run_config.json'):
        if all((path.parent / n).is_file() for n in ['best.pt', 'tokenizer.json']):
            kind = classify_model_config(read_json(path).get('model', {}))
            if kind:
                candidates[kind].append(path.parent)
    for name in MODEL_NAMES:
        if MODEL_DIRS[name] is None:
            options = sorted(set(candidates[name]))
            if len(options) != 1:
                raise ValueError(f'Set MODEL_DIRS[{name!r}] to one checkpoint folder. Found: {options}')
            MODEL_DIRS[name] = options[0]
        MODEL_DIRS[name] = Path(MODEL_DIRS[name])
        for filename in ('best.pt', 'tokenizer.json', 'run_config.json'):
            if not (MODEL_DIRS[name] / filename).is_file():
                raise FileNotFoundError(MODEL_DIRS[name] / filename)

    names = ['Train/train.json', 'Train/dev.json', 'Test/test.json']
    def valid(root): return all((root / n).is_file() for n in names)
    if DATA_ROOT is not None:
        source = Path(DATA_ROOT)
        if not valid(source): raise FileNotFoundError(f'Incomplete CJPE folder: {source}')
    else:
        found = sorted(set(p.parent.parent for p in INPUT_ROOT.rglob('train.json')
                           if p.parent.name == 'Train' and valid(p.parent.parent)))
        saved = ASSET_DIR / 'Phase_4_CJPE'
        if valid(saved): found = [saved]
        if len(found) > 1: raise ValueError(f'Set DATA_ROOT explicitly: {found}')
        if found:
            source = found[0]
        else:
            from huggingface_hub import snapshot_download
            try:
                snapshot = Path(snapshot_download(repo_id=REPO, repo_type='dataset',
                    revision=DATA_REVISION, allow_patterns=['Phase_4_CJPE/**']))
            except Exception as error:
                raise RuntimeError('Attach Phase_4_CJPE or enable Internet and add an authorized HF_TOKEN secret.') from error
            source = snapshot / 'Phase_4_CJPE'
            if not valid(source): raise FileNotFoundError(f'CJPE files missing under {snapshot}')
    destination = ASSET_DIR / 'Phase_4_CJPE'
    for name in names:
        target = destination / name
        target.parent.mkdir(parents=True, exist_ok=True)
        if source.resolve() != destination.resolve():
            shutil.copy2(source / name, target)
    print('Data:', destination)
    for name, folder in MODEL_DIRS.items(): print(name, folder)
    return destination

DATA_ROOT = resolve_inputs()
write_json(ASSET_DIR / 'environment.json', {
    'version': VERSION, 'torch': torch.__version__,
    'packages': {p: importlib.metadata.version(p) for p in
                 ['peft', 'transformers', 'huggingface-hub', 'scikit-learn', 'ijson']},
    'data_hashes': {n: digest_file(DATA_ROOT / n) for n in
                    ['Train/train.json', 'Train/dev.json', 'Test/test.json']},
})


Data: /kaggle/working/phase4_cjpe/assets/Phase_4_CJPE
model1 /kaggle/input/datasets/acadsiitk/phase3-req/phase3_model_assets/model1_decoder
model2 /kaggle/input/datasets/acadsiitk/phase3-req/phase3_model_assets/model2_encoder_decoder
model3 /kaggle/input/datasets/acadsiitk/phase3-req/phase3_model_assets/model3_local_global
model4 /kaggle/input/datasets/acadsiitk/phase3-req/phase3_model_assets/model4_gated_cnn


## 3. Audit the corpus

The audit collapses identical repeated case IDs and stops on conflicting copies. Training excludes cases whose text matches development or test text.

In [ ]:
def iter_records(path):
    with open(path, 'rb') as f:
        first = f.read(4096).lstrip(b'\xef\xbb\xbf \r\n\t')[:1]
    with open(path, 'rb') as f:
        if first == b'[':
            yield from ijson.items(f, 'item')
        elif first == b'{':
            for key, value in ijson.kvitems(f, ''):
                if not isinstance(value, dict):
                    raise ValueError(f'{path}: expected records keyed by case ID, found {key!r}')
                record = dict(value)
                record.setdefault('id', key)
                yield record
        else:
            raise ValueError(f'{path}: expected a JSON list or keyed dictionary')

def clean_record(record, labelled):
    if 'id' not in record or 'text' not in record:
        raise ValueError(f'Record needs id/text. Available keys: {list(record)}')
    identifier = record['id']
    if not isinstance(identifier, str) or not identifier.strip():
        raise ValueError(f'Expected a nonempty string case ID, got {identifier!r}')
    text = record['text']
    if not isinstance(text, str) or not text.strip():
        raise ValueError(f'Case {identifier}: text must be a nonempty string')
    label = None
    if labelled:
        keys = [LABEL_FIELD] if LABEL_FIELD else [k for k in ('label','gold','outcome','judgment') if k in record]
        if len(keys) != 1 or keys[0] not in record:
            raise ValueError(f'Set LABEL_FIELD. Case {identifier} has keys {list(record)}')
        raw = record[keys[0]]
        if isinstance(raw, bool) or raw not in (0, 1, '0', '1'):
            raise ValueError(f'Case {identifier}: unsupported binary label {raw!r}')
        label = int(raw)
    fingerprint = hashlib.sha256(' '.join(text.split()).encode('utf-8')).hexdigest()
    return identifier, text, label, fingerprint

def audit_corpus():
    index, report, exclusions = {}, {}, []
    blocked = set()
    for split in ('dev','test','train'):
        path = DATA_ROOT / ('Test/test.json' if split == 'test' else f'Train/{split}.json')
        rows, ids, seen_text, labels, lengths = [], set(), {}, Counter(), []
        for record in tqdm(iter_records(path), desc=f'Audit {split}'):
            identifier, text, label, text_hash = clean_record(record, split != 'test')
            if identifier in ids: raise ValueError(f'Duplicate ID in {split}: {identifier}')
            ids.add(identifier)
            if split == 'train':
                if text_hash in seen_text and seen_text[text_hash] != label:
                    raise ValueError(f'Conflicting training labels for identical text: {identifier}')
                repeated = text_hash in seen_text
                seen_text[text_hash] = label
                if text_hash in blocked or repeated:
                    exclusions.append({'id': identifier, 'reason': 'heldout_text' if text_hash in blocked else 'duplicate_train_text'})
                    continue
            else:
                blocked.add(text_hash)
            rows.append(dict(id=identifier, label=label, text_sha256=text_hash))
            labels[str(label)] += 1
            lengths.append(len(text.split()))
        if not rows: raise ValueError(f'Empty {split} split')
        index[split] = rows
        report[split] = dict(examples=len(rows), labels=dict(labels),
            word_percentiles=dict(zip(['50','90','95','99','100'], np.percentile(lengths,[50,90,95,99,100]).tolist())))
    if set(r['label'] for r in index['train']) != {0,1}:
        raise ValueError('Training must contain both classes')
    report['excluded_training_records'] = len(exclusions)
    write_json(ASSET_DIR / 'corpus_audit.json', report)
    write_json(ASSET_DIR / 'excluded_train.json', exclusions)
    write_json(ASSET_DIR / 'case_index.json', index)
    write_json(ASSET_DIR / 'dev_truth.json', [dict(id=r['id'],gold=r['label']) for r in index['dev']])
    print(json.dumps(report, indent=2))
    return index

In [9]:
import hashlib
from pathlib import Path

if "_phase4_original_iter_records" not in globals():
    _phase4_original_iter_records = iter_records



def iter_records(path):
    path = Path(path)
    labelled = path.stem.lower() != "test"
    seen = {}
    duplicates = 0

    for record in _phase4_original_iter_records(path):
        identifier, text, label, _ = clean_record(record, labelled)
        text_hash = hashlib.sha256(text.encode("utf-8")).hexdigest()
        identity = (text_hash, label)

        if identifier in seen:
            first = seen[identifier]

            if first["identity"] != identity:
                details = {
                    "file": str(path),
                    "id": identifier,
                    "first_label": first["identity"][1],
                    "repeated_label": label,
                    "same_text": first["identity"][0] == text_hash,
                    "first_characters": first["characters"],
                    "repeated_characters": len(text),
                    "first_preview": first["preview"],
                    "repeated_preview": text[:200],
                }
                print(details)

                raise ValueError(
                    f"{identifier}: repeated ID has different text or labels. "
                    "No conflicting record was discarded. "
                    "Share the details printed above."
                )

            duplicates += 1
            continue

        seen[identifier] = {
            "identity": identity,
            "characters": len(text),
            "preview": text[:200],
        }
        yield record

    print(
        f"{path.name}: {len(seen):,} unique IDs; "
        f"{duplicates:,} identical repeated records skipped."
    )


CASE_INDEX = audit_corpus()

Audit dev: 0it [00:00, ?it/s]

dev.json: 2,511 unique IDs; 994 identical repeated records skipped.


Audit test: 0it [00:00, ?it/s]

test.json: 1,517 unique IDs; 0 identical repeated records skipped.


Audit train: 0it [00:00, ?it/s]

train.json: 32,305 unique IDs; 5,082 identical repeated records skipped.
{
  "dev": {
    "examples": 2511,
    "labels": {
      "0": 1252,
      "1": 1259
    },
    "word_percentiles": {
      "50": 2909.0,
      "90": 7210.0,
      "95": 9634.5,
      "99": 18266.100000000002,
      "100": 42198.0
    }
  },
  "test": {
    "examples": 1517,
    "labels": {
      "None": 1517
    },
    "word_percentiles": {
      "50": 2940.0,
      "90": 7302.400000000001,
      "95": 10023.400000000001,
      "99": 18235.599999999988,
      "100": 40487.0
    }
  },
  "train": {
    "examples": 31653,
    "labels": {
      "0": 18539,
      "1": 13114
    },
    "word_percentiles": {
      "50": 2030.0,
      "90": 6430.799999999999,
      "95": 8986.399999999998,
      "99": 19272.759999999995,
      "100": 412606.0
    }
  },
  "excluded_training_records": 652
}


## 4. Original tokenizer and backbone definitions

These definitions match the Phase 1 checkpoints. Classification heads are new. Libraries handle adaptation where supported; no external pretrained model is loaded.


In [10]:
def split_bpe_pieces(text, max_chars=256):
    """Keep every character; separate words, punctuation and whitespace.

    An optional leading ASCII space can join a word or punctuation piece.
    Long pieces are split into bounded chunks to limit per-piece work.
    No normalization, strip(), split(), external tokenizer or regex package.
    """
    for match in re.finditer(' ?\\w+| ?[^\\w\\s]+|\\s+', text):
        piece = match.group(0)
        for start in range(0, len(piece), max_chars):
            yield piece[start:start + max_chars].encode('utf-8')


def apply_bpe_merges(piece, merge_ranks):
    """Apply learned merges by rank, breaking occurrence ties left-to-right.

    Token IDs 0..3 are reserved. Byte b always has ID b+4. A linked list
    and a heap avoid repeatedly scanning every adjacent pair in a piece.
    """
    ids = [byte + 4 for byte in piece]
    size = len(ids)
    if size < 2 or not merge_ranks:
        return tuple(ids)
    previous = [i - 1 for i in range(size)]
    following = [i + 1 for i in range(size)]
    following[-1] = -1
    active = [True] * size
    queue = []

    def offer(left):
        if left < 0 or not active[left]:
            return
        right = following[left]
        if right < 0:
            return
        pair = (ids[left], ids[right])
        rank = merge_ranks.get(pair)
        if rank is not None:
            heapq.heappush(queue, (rank, left, right, pair[0], pair[1]))
    for left in range(size - 1):
        offer(left)
    while queue:
        rank, left, right, a, b = heapq.heappop(queue)
        if not active[left] or not active[right] or following[left] != right:
            continue
        if ids[left] != a or ids[right] != b:
            continue
        ids[left] = 260 + rank
        active[right] = False
        following[left] = following[right]
        if following[right] >= 0:
            previous[following[right]] = left
        offer(previous[left])
        offer(left)
    result = []
    position = 0
    while position >= 0:
        result.append(ids[position])
        position = following[position]
    return tuple(result)


class ScratchByteBPE:
    """Our byte BPE vocabulary, encoder, decoder and JSON serialization.

    All base bytes and EVERY intermediate merge token are retained.
    Integer IDs identify symbols; concatenated decimal strings are never used.
    Literal '<s>' etc. in ordinary input are encoded as text, not control IDs.
    """
    FORMAT = 'handwritten-byte-bpe-v1'
    PATTERN = ' ?\\w+| ?[^\\w\\s]+|\\s+'

    def __init__(self, merges=None, max_piece_chars=256):
        self.specials = ['<pad>', '<unk>', '<s>', '</s>']
        self.max_piece_chars = max_piece_chars
        if not isinstance(max_piece_chars, int) or max_piece_chars < 1:
            raise ValueError('max_piece_chars must be a positive integer')
        self.merges = []
        self.merge_ranks = {}
        self.token_bytes = [None] * 4 + [bytes([b]) for b in range(256)]
        self._cache = {}
        for pair in merges or []:
            self.add_merge(tuple(pair))

    def add_merge(self, pair):
        if len(pair) != 2 or pair in self.merge_ranks:
            raise ValueError('Invalid or repeated merge pair')
        if any((type(token) is not int or not 4 <= token < len(self.token_bytes) for token in pair)):
            raise ValueError('A merge must reference earlier non-special token IDs')
        new_id = len(self.token_bytes)
        self.merge_ranks[pair] = len(self.merges)
        self.merges.append(pair)
        self.token_bytes.append(self.token_bytes[pair[0]] + self.token_bytes[pair[1]])
        self._cache.clear()
        return new_id

    def get_vocab_size(self):
        return len(self.token_bytes)

    def token_to_id(self, special):
        return self.specials.index(special) if special in self.specials else None

    def encode(self, text):
        if not isinstance(text, str):
            raise TypeError('encode expects a string')
        output = []
        for piece in split_bpe_pieces(text, self.max_piece_chars):
            encoded = self._cache.get(piece)
            if encoded is None:
                encoded = apply_bpe_merges(piece, self.merge_ranks)
                if len(self._cache) >= 65536:
                    self._cache.clear()
                self._cache[piece] = encoded
            output.extend(encoded)
        return SimpleNamespace(ids=output)

    def decode(self, ids, skip_special_tokens=True):
        chunks = []
        for value in ids:
            token = int(value)
            if not 0 <= token < len(self.token_bytes):
                raise ValueError(f'Unknown token ID: {token}')
            if token < 4:
                if not skip_special_tokens:
                    chunks.append(self.specials[token].encode('utf-8'))
            else:
                chunks.append(self.token_bytes[token])
        return b''.join(chunks).decode('utf-8', errors='replace')

    def save(self, path, training=None):
        payload = {'format': self.FORMAT, 'special_tokens': self.specials, 'byte_offset': 4, 'pattern': self.PATTERN, 'max_piece_chars': self.max_piece_chars, 'merges': self.merges, 'vocab_hex': [None if value is None else value.hex() for value in self.token_bytes]}
        if training is not None:
            payload['training'] = training
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_suffix(path.suffix + '.tmp')
        temporary.write_text(json.dumps(payload, ensure_ascii=False), encoding='utf-8')
        temporary.replace(path)

    @classmethod
    def from_file(cls, path):
        payload = json.loads(Path(path).read_text(encoding='utf-8'))
        if payload.get('format') != cls.FORMAT:
            raise ValueError('This is not our handwritten BPE format. Use a fresh data directory and retrain the tokenizer/token arrays.')
        if payload.get('special_tokens') != ['<pad>', '<unk>', '<s>', '</s>']:
            raise ValueError('Special-token IDs do not match')
        if payload.get('byte_offset') != 4 or payload.get('pattern') != cls.PATTERN:
            raise ValueError('Byte mapping or splitting rules do not match')
        tokenizer = cls(payload['merges'], payload['max_piece_chars'])
        expected = [None if value is None else value.hex() for value in tokenizer.token_bytes]
        if payload.get('vocab_hex') != expected:
            raise ValueError('Vocabulary is inconsistent with the merge history')
        return tokenizer


In [11]:
class DecoderAttention(nn.Module):

    def __init__(self, width, heads, dropout):
        super().__init__()
        assert width % heads == 0
        self.heads, self.dim, self.dropout = (heads, width // heads, dropout)
        self.qkv = nn.Linear(width, 3 * width)
        self.proj = nn.Linear(width, width)

    def forward(self, x, past=None, cache=False):
        b, t, c = x.shape
        q, k, v = self.qkv(x).view(b, t, 3, self.heads, self.dim).unbind(2)
        q, k, v = (a.transpose(1, 2) for a in (q, k, v))
        if past is not None:
            if t != 1:
                raise ValueError('Cached decoding accepts one new token at a time')
            k, v = (torch.cat((past[0], k), 2), torch.cat((past[1], v), 2))
        y = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=past is None)
        return (self.proj(y.transpose(1, 2).contiguous().view(b, t, c)), (k, v) if cache else None)


class Block(nn.Module):

    def __init__(self, width, heads, dropout):
        super().__init__()
        self.ln1, self.ln2 = (nn.LayerNorm(width), nn.LayerNorm(width))
        self.attn = DecoderAttention(width, heads, dropout)
        self.mlp = nn.Sequential(nn.Linear(width, 4 * width), nn.GELU(), nn.Linear(4 * width, width))
        self.drop = nn.Dropout(dropout)

    def forward(self, x, past=None, cache=False):
        a, kv = self.attn(self.ln1(x), past, cache)
        x = x + self.drop(a)
        return (x + self.drop(self.mlp(self.ln2(x))), kv)


class DecoderLM(nn.Module):

    def __init__(self, vocab_size=32768, width=704, layers=16, heads=11, context=2048, dropout=0.0):
        super().__init__()
        self.config = dict(vocab_size=vocab_size, width=width, layers=layers, heads=heads, context=context, dropout=dropout)
        self.token = nn.Embedding(vocab_size, width)
        self.position = nn.Embedding(context, width)
        self.blocks = nn.ModuleList([Block(width, heads, dropout) for _ in range(layers)])
        self.final_norm = nn.LayerNorm(width)
        self.drop = nn.Dropout(dropout)
        self.gradient_checkpointing = True
        self.apply(self._init)
        for block in self.blocks:
            nn.init.normal_(block.attn.proj.weight, std=0.02 / math.sqrt(2 * layers))
            nn.init.normal_(block.mlp[2].weight, std=0.02 / math.sqrt(2 * layers))

    @staticmethod
    def _init(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, ids, targets=None, past=None, cache=False):
        offset = 0 if past is None else past[0][0].shape[2]
        if offset + ids.shape[1] > self.config['context']:
            raise ValueError('Context window exceeded')
        x = self.drop(self.token(ids) + self.position(torch.arange(offset, offset + ids.shape[1], device=ids.device)))
        kvs = []
        for i, block in enumerate(self.blocks):
            if self.training and self.gradient_checkpointing:
                x = checkpoint(lambda z, b=block: b(z)[0], x, use_reentrant=False)
            else:
                x, kv = block(x, None if past is None else past[i], cache)
                if cache:
                    kvs.append(kv)
        x = self.final_norm(x)
        if targets is not None:
            flat, labels = (x.reshape(-1, x.shape[-1]), targets.reshape(-1))

            def chunk_loss(h, y):
                return F.cross_entropy(F.linear(h, self.token.weight).float(), y, ignore_index=-100, reduction='sum')
            loss = x.new_zeros((), dtype=torch.float32)
            for start in range(0, len(labels), 256):
                h, y = (flat[start:start + 256], labels[start:start + 256])
                loss = loss + (checkpoint(chunk_loss, h, y, use_reentrant=False) if self.training else chunk_loss(h, y))
            return loss
        return (F.linear(x[:, -1], self.token.weight), kvs)


In [12]:
class EncDecAttention(nn.Module):
    def __init__(self,width,heads,dropout=0.0):
        super().__init__(); assert width%heads==0
        self.heads=heads; self.dim=width//heads; self.dropout=dropout
        self.q=nn.Linear(width,width); self.k=nn.Linear(width,width)
        self.v=nn.Linear(width,width); self.out=nn.Linear(width,width)

    def forward(self,x,memory=None,valid=None,causal=False,past=None,cache=False):
        b,t,w=x.shape
        def split(z): return z.view(b,-1,self.heads,self.dim).transpose(1,2)
        q=split(self.q(x))
        if memory is not None and past is not None:
            k,v=past
        else:
            source=x if memory is None else memory
            k,v=split(self.k(source)),split(self.v(source))
            if past is not None:
                k=torch.cat((past[0],k),dim=2); v=torch.cat((past[1],v),dim=2)
        mask=None
        if valid is not None: mask=valid[:,None,None,:]
        if causal:
            offset=k.shape[2]-t
            allowed=torch.arange(k.shape[2],device=x.device)[None,:] <= (torch.arange(t,device=x.device)+offset)[:,None]
            mask=allowed[None,None,:,:] if mask is None else mask & allowed[None,None,:,:]
        h=F.scaled_dot_product_attention(q,k,v,attn_mask=mask,
            dropout_p=self.dropout if self.training else 0.0)
        h=self.out(h.transpose(1,2).contiguous().view(b,t,w))
        return h, (k,v) if cache else None

class EncoderBlock(nn.Module):
    def __init__(self,w,h,p):
        super().__init__(); self.n1=nn.LayerNorm(w); self.n2=nn.LayerNorm(w)
        self.attn=EncDecAttention(w,h,p); self.drop=nn.Dropout(p)
        self.ff=nn.Sequential(nn.Linear(w,4*w),nn.GELU(),nn.Linear(4*w,w))
    def forward(self,x,valid):
        x=x+self.drop(self.attn(self.n1(x),valid=valid)[0])
        return x+self.drop(self.ff(self.n2(x)))

class DecoderBlock(nn.Module):
    def __init__(self,w,h,p):
        super().__init__(); self.n1=nn.LayerNorm(w); self.n2=nn.LayerNorm(w); self.n3=nn.LayerNorm(w)
        self.self_attn=EncDecAttention(w,h,p); self.cross=EncDecAttention(w,h,p); self.drop=nn.Dropout(p)
        self.ff=nn.Sequential(nn.Linear(w,4*w),nn.GELU(),nn.Linear(4*w,w))
    def forward(self,x,memory,source_valid,target_valid=None,past=None,cache=False):
        self_past,cross_past=(None,None) if past is None else past
        a,skv=self.self_attn(self.n1(x),valid=target_valid,causal=True,past=self_past,cache=cache)
        x=x+self.drop(a)
        a,ckv=self.cross(self.n2(x),memory=memory,valid=source_valid,past=cross_past,cache=cache)
        x=x+self.drop(a); x=x+self.drop(self.ff(self.n3(x)))
        return x, (skv,ckv) if cache else None


class EncoderDecoder(nn.Module):
    def __init__(self,vocab_size=32768,width=704,encoder_layers=6,decoder_layers=8,heads=11,dropout=0.0):
        super().__init__()
        self.config=dict(vocab_size=vocab_size,width=width,encoder_layers=encoder_layers,
                         decoder_layers=decoder_layers,heads=heads,dropout=dropout)
        self.token=nn.Embedding(vocab_size,width)
        self.encoder=nn.ModuleList([EncoderBlock(width,heads,dropout) for _ in range(encoder_layers)])
        self.decoder=nn.ModuleList([DecoderBlock(width,heads,dropout) for _ in range(decoder_layers)])
        self.enc_norm=nn.LayerNorm(width); self.dec_norm=nn.LayerNorm(width)
        self.drop=nn.Dropout(dropout); self.gradient_checkpointing=True
        self.apply(self.initialize)

    @staticmethod
    def initialize(m):
        if isinstance(m,(nn.Linear,nn.Embedding)):
            nn.init.normal_(m.weight,std=0.02)
            if isinstance(m,nn.Linear) and m.bias is not None: nn.init.zeros_(m.bias)

    def embed(self,ids,offset=0):
        w=self.config['width']; positions=torch.arange(offset,offset+ids.shape[1],device=ids.device).float()
        frequency=torch.exp(torch.arange(0,w,2,device=ids.device).float()*(-math.log(10000)/w))
        angle=positions[:,None]*frequency[None,:]
        pe=torch.stack((angle.sin(),angle.cos()),dim=-1).flatten(1)[:,:w]
        return self.drop(self.token(ids)+pe.to(self.token.weight.dtype))

    def encode(self,src):
        valid=src.ne(PAD); x=self.embed(src)
        for block in self.encoder:
            if self.training and self.gradient_checkpointing:
                x=checkpoint(block,x,valid,use_reentrant=False)
            else: x=block(x,valid)
        return self.enc_norm(x),valid

    def decode(self,ids,memory,source_valid,past=None,cache=False):
        offset=0 if past is None else past[0][0][0].shape[2]
        x=self.embed(ids,offset)
        valid=ids.ne(PAD) if past is None else None
        caches=[]
        for i,block in enumerate(self.decoder):
            if self.training and self.gradient_checkpointing:
                x=checkpoint(lambda a,b,c,d,layer=block: layer(a,b,c,d)[0],
                    x,memory,source_valid,valid,use_reentrant=False)
            else:
                x,kv=block(x,memory,source_valid,valid,None if past is None else past[i],cache)
                if cache: caches.append(kv)
        return self.dec_norm(x),caches

    def forward(self,src,decoder_ids,labels=None):
        memory,valid=self.encode(src); h,_=self.decode(decoder_ids,memory,valid)
        if labels is None: return F.linear(h,self.token.weight)
        flat=h.reshape(-1,h.shape[-1]); labels=labels.reshape(-1)
        def loss_chunk(a,b):
            return F.cross_entropy(F.linear(a,self.token.weight).float(),b,ignore_index=-100,reduction='sum')
        loss=h.new_zeros((),dtype=torch.float32)
        for start in range(0,len(labels),128):
            a,b=flat[start:start+128],labels[start:start+128]
            loss=loss+(checkpoint(loss_chunk,a,b,use_reentrant=False) if self.training else loss_chunk(a,b))
        return loss


In [13]:
def sliding_attention(q,k,v,window,chunk,dropout=0.0):
    b,h,t,d=q.shape; blocks=math.ceil(t/chunk); padded=blocks*chunk
    qpad=F.pad(q,(0,0,0,padded-t))
    query=qpad.reshape(b,h,blocks,chunk,d).permute(0,2,1,3,4).reshape(b*blocks,h,chunk,d)
    positions=torch.arange(blocks,device=q.device)[:,None]*chunk-window+1+torch.arange(window+chunk-1,device=q.device)[None,:]
    indices=positions.clamp(0,t-1)
    keys=k[:,:,indices,:].permute(0,2,1,3,4).reshape(b*blocks,h,window+chunk-1,d)
    values=v[:,:,indices,:].permute(0,2,1,3,4).reshape(b*blocks,h,window+chunk-1,d)
    qi=torch.arange(chunk,device=q.device)[:,None]
    kj=torch.arange(window+chunk-1,device=q.device)[None,:]
    relative=(kj>=qi)&(kj<=qi+window-1)
    valid=(positions>=0)&(positions<t)
    mask=(relative[None,:,:]&valid[:,None,:])[None,:,:,:].expand(b,-1,-1,-1).reshape(b*blocks,1,chunk,window+chunk-1)
    output=F.scaled_dot_product_attention(query,keys,values,attn_mask=mask,dropout_p=dropout)
    return output.reshape(b,blocks,h,chunk,d).permute(0,2,1,3,4).reshape(b,h,padded,d)[:,:,:t]

class MixedAttention(nn.Module):
    def __init__(self,width,heads,window,query_chunk,global_layer,dropout):
        super().__init__(); assert width%heads==0
        self.heads=heads; self.dim=width//heads; self.window=window
        self.chunk=query_chunk; self.global_layer=global_layer; self.dropout=dropout
        self.qkv=nn.Linear(width,3*width); self.proj=nn.Linear(width,width)
    def forward(self,x,past=None,cache=False):
        b,t,w=x.shape
        q,k,v=self.qkv(x).view(b,t,3,self.heads,self.dim).unbind(2)
        q,k,v=(a.transpose(1,2) for a in (q,k,v))
        if past is not None:
            if t!=1: raise ValueError('Incremental decoding expects one token')
            k=torch.cat((past[0],k),dim=2); v=torch.cat((past[1],v),dim=2)
        p=self.dropout if self.training else 0.0
        if self.global_layer or past is not None:
            output=F.scaled_dot_product_attention(q,k,v,dropout_p=p,is_causal=past is None)
        else: output=sliding_attention(q,k,v,self.window,self.chunk,p)
        kv=None
        if cache:
            kv=(k,v) if self.global_layer else (k[:,:,-(self.window-1):].contiguous(),v[:,:,-(self.window-1):].contiguous())
        return self.proj(output.transpose(1,2).contiguous().view(b,t,w)),kv

class MixedBlock(nn.Module):
    def __init__(self,width,heads,window,query_chunk,global_layer,dropout):
        super().__init__(); self.n1=nn.LayerNorm(width); self.n2=nn.LayerNorm(width)
        self.attn=MixedAttention(width,heads,window,query_chunk,global_layer,dropout)
        self.ff=nn.Sequential(nn.Linear(width,4*width),nn.GELU(),nn.Linear(4*width,width))
        self.drop=nn.Dropout(dropout)
    def forward(self,x,past=None,cache=False):
        a,kv=self.attn(self.n1(x),past,cache); x=x+self.drop(a)
        return x+self.drop(self.ff(self.n2(x))),kv


class LocalGlobalLM(nn.Module):
    def __init__(self,vocab_size=32768,width=704,layers=16,heads=11,context=2048,window=256,
                 global_every=4,query_chunk=128,dropout=0.0,activation_checkpointing=True,loss_chunk=256):
        super().__init__()
        if min(context,query_chunk,global_every,loss_chunk)<1 or window<2: raise ValueError('Invalid architecture limits')
        self.config=dict(vocab_size=vocab_size,width=width,layers=layers,heads=heads,context=context,
            window=window,global_every=global_every,query_chunk=query_chunk,dropout=dropout,
            activation_checkpointing=activation_checkpointing,loss_chunk=loss_chunk)
        self.token=nn.Embedding(vocab_size,width)
        self.blocks=nn.ModuleList([MixedBlock(width,heads,window,query_chunk,(i+1)%global_every==0,dropout) for i in range(layers)])
        self.norm=nn.LayerNorm(width); self.drop=nn.Dropout(dropout)
        self.apply(self.initialize)
        for block in self.blocks:
            nn.init.normal_(block.attn.proj.weight,std=0.02/math.sqrt(2*layers))
            nn.init.normal_(block.ff[2].weight,std=0.02/math.sqrt(2*layers))
    @staticmethod
    def initialize(m):
        if isinstance(m,(nn.Linear,nn.Embedding)):
            nn.init.normal_(m.weight,std=0.02)
            if isinstance(m,nn.Linear) and m.bias is not None: nn.init.zeros_(m.bias)
    def hidden(self,ids,past=None,position_offset=0,cache=False):
        t=ids.shape[1]; w=self.config['width']
        if position_offset+t>self.config['context']: raise ValueError('Context exceeded')
        pos=torch.arange(position_offset,position_offset+t,device=ids.device).float()
        freq=torch.exp(torch.arange(0,w,2,device=ids.device).float()*(-math.log(10000)/w))
        phase=pos[:,None]*freq[None,:]
        pe=torch.stack((phase.sin(),phase.cos()),dim=-1).flatten(1)[:,:w]
        # Match sinusoidal amplitude with scaled learned token embeddings.
        x=self.drop(self.token(ids)*math.sqrt(w)+pe.to(self.token.weight.dtype))
        caches=[]
        for i,block in enumerate(self.blocks):
            if self.training and self.config['activation_checkpointing']:
                x=checkpoint(lambda z,layer=block: layer(z)[0],x,use_reentrant=False)
            else:
                x,kv=block(x,None if past is None else past[i],cache)
                if cache: caches.append(kv)
        return self.norm(x),caches
    def forward(self,ids,targets=None,past=None,position_offset=0,cache=False):
        h,caches=self.hidden(ids,past,position_offset,cache)
        if targets is None: return F.linear(h[:,-1],self.token.weight),caches
        flat=h.reshape(-1,h.shape[-1]); labels=targets.reshape(-1)
        def chunk_loss(a,b): return F.cross_entropy(F.linear(a,self.token.weight).float(),b,ignore_index=-100,reduction='sum')
        loss=h.new_zeros((),dtype=torch.float32); size=self.config['loss_chunk']
        for start in range(0,len(labels),size):
            a,b=flat[start:start+size],labels[start:start+size]
            loss=loss+(checkpoint(chunk_loss,a,b,use_reentrant=False) if self.training and self.config['activation_checkpointing'] else chunk_loss(a,b))
        return loss


In [14]:
class CausalGatedConv(nn.Module):
    def __init__(self,width,kernel_size,dilation):
        super().__init__()
        if kernel_size<2 or dilation<1: raise ValueError('Invalid causal convolution settings')
        self.history=(kernel_size-1)*dilation
        self.conv=nn.Conv1d(width,2*width,kernel_size,dilation=dilation,padding=0)
        self.proj=nn.Linear(width,width)

    def forward(self,x,past=None,cache=False):
        values=x.transpose(1,2)
        if past is None:
            padded=F.pad(values,(self.history,0))
        else:
            if past.shape!=(x.shape[0],x.shape[2],self.history):
                raise ValueError('Convolution cache shape mismatch')
            padded=torch.cat((past,values),dim=2)
        features=self.conv(padded)
        a,b=features.chunk(2,dim=1)
        output=self.proj((a*torch.sigmoid(b)).transpose(1,2))
        # clone prevents a small cache view from retaining a full prompt buffer.
        state=padded[:,:,-self.history:].clone() if cache else None
        return output,state

class ConvBlock(nn.Module):
    def __init__(self,width,kernel_size,dilation,dropout):
        super().__init__()
        self.n1=nn.LayerNorm(width); self.n2=nn.LayerNorm(width)
        self.causal=CausalGatedConv(width,kernel_size,dilation)
        self.ff=nn.Sequential(nn.Linear(width,4*width),nn.GELU(),nn.Linear(4*width,width))
        self.drop=nn.Dropout(dropout)

    def forward(self,x,past=None,cache=False):
        h,state=self.causal(self.n1(x),past,cache)
        x=x+self.drop(h)
        return x+self.drop(self.ff(self.n2(x))),state


class GatedConvLM(nn.Module):
    def __init__(self,vocab_size=32768,width=640,layers=16,kernel_size=3,dilation_cycle=10,
                 context=2048,dropout=0.0,activation_checkpointing=True,loss_chunk=256):
        super().__init__()
        if min(width,layers,dilation_cycle,context,loss_chunk)<1 or kernel_size<2:
            raise ValueError('Invalid model configuration')
        self.config=dict(vocab_size=vocab_size,width=width,layers=layers,kernel_size=kernel_size,
            dilation_cycle=dilation_cycle,context=context,dropout=dropout,
            activation_checkpointing=activation_checkpointing,loss_chunk=loss_chunk)
        self.dilations=[2**(i%dilation_cycle) for i in range(layers)]
        self.receptive_field=1+(kernel_size-1)*sum(self.dilations)
        self.token=nn.Embedding(vocab_size,width)
        self.blocks=nn.ModuleList([ConvBlock(width,kernel_size,d,dropout) for d in self.dilations])
        self.norm=nn.LayerNorm(width); self.drop=nn.Dropout(dropout)
        self.apply(self.initialize)
        for block in self.blocks:
            nn.init.normal_(block.causal.proj.weight,std=0.02/math.sqrt(2*layers))
            nn.init.normal_(block.ff[2].weight,std=0.02/math.sqrt(2*layers))

    @staticmethod
    def initialize(module):
        if isinstance(module,(nn.Linear,nn.Conv1d,nn.Embedding)):
            nn.init.normal_(module.weight,std=0.02)
            if getattr(module,'bias',None) is not None: nn.init.zeros_(module.bias)

    def hidden(self,ids,past=None,cache=False):
        if past is None and ids.shape[1]>self.config['context']:
            raise ValueError('Initial prompt/training block exceeds configured context')
        if past is not None and (len(past)!=len(self.blocks) or ids.shape[1]!=1):
            raise ValueError('Cached generation accepts one token and one state per layer')
        if self.training and (cache or past is not None):
            raise ValueError('Caches are for eval-mode generation only')
        x=self.drop(self.token(ids)); states=[]
        for i,block in enumerate(self.blocks):
            if self.training and self.config['activation_checkpointing']:
                x=checkpoint(lambda z,layer=block: layer(z)[0],x,use_reentrant=False)
            else:
                x,state=block(x,None if past is None else past[i],cache)
                if cache: states.append(state)
        return self.norm(x),states

    def forward(self,ids,targets=None,past=None,cache=False):
        h,states=self.hidden(ids,past,cache)
        if targets is None: return F.linear(h[:,-1],self.token.weight),states
        flat=h.reshape(-1,h.shape[-1]); labels=targets.reshape(-1)
        def chunk_loss(a,b):
            return F.cross_entropy(F.linear(a,self.token.weight).float(),b,ignore_index=-100,reduction='sum')
        loss=h.new_zeros((),dtype=torch.float32); size=self.config['loss_chunk']
        for start in range(0,len(labels),size):
            a,b=flat[start:start+size],labels[start:start+size]
            loss=loss+(checkpoint(chunk_loss,a,b,use_reentrant=False) if self.training and self.config['activation_checkpointing'] else chunk_loss(a,b))
        return loss


In [15]:
MODEL_CLASSES = {
    "model1": DecoderLM,
    "model2": EncoderDecoder,
    "model3": LocalGlobalLM,
    "model4": GatedConvLM,
}

BASES = {}
for name in MODEL_NAMES:
    folder = MODEL_DIRS[name]
    checkpoint_path = folder / "best.pt"
    tokenizer_path = folder / "tokenizer.json"
    state = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    config = copy.deepcopy(state["model_config"])
    detected = classify_model_config(config)
    if detected != name:
        raise ValueError(f"{folder} is {detected}, expected {name}")
    tokenizer = ScratchByteBPE.from_file(tokenizer_path)
    if config["vocab_size"] != tokenizer.get_vocab_size():
        raise ValueError(f"Vocabulary mismatch for {name}")
    expected_hash = state.get("tokenizer_sha256")
    if expected_hash and digest_file(tokenizer_path) != expected_hash:
        raise ValueError(f"Tokenizer hash mismatch for {name}")
    BASES[name] = {
        "folder": folder,
        "checkpoint": checkpoint_path,
        "tokenizer_path": tokenizer_path,
        "tokenizer": tokenizer,
        "config": config,
        "state": state["model"],
        "checkpoint_sha256": digest_file(checkpoint_path),
        "tokenizer_sha256": digest_file(tokenizer_path),
    }

base_report = {}
for name in MODEL_NAMES:
    model = MODEL_CLASSES[name](**BASES[name]["config"])
    result = model.load_state_dict(BASES[name]["state"], strict=True)
    assert not result.missing_keys and not result.unexpected_keys
    count = sum(parameter.numel() for parameter in model.parameters())
    base_report[name] = {"parameters": count, "config": BASES[name]["config"]}
    print(name, f"{count:,}", BASES[name]["config"])
    del model
    gc.collect()
write_json(ASSET_DIR / "phase1_models.json", base_report)


model1 119,816,576 {'vocab_size': 32768, 'width': 704, 'layers': 16, 'heads': 11, 'context': 2048, 'dropout': 0.0}
model2 122,356,608 {'vocab_size': 32768, 'width': 704, 'encoder_layers': 6, 'decoder_layers': 8, 'heads': 11, 'dropout': 0.0}
model3 118,374,784 {'vocab_size': 32768, 'width': 704, 'layers': 16, 'heads': 11, 'context': 2048, 'window': 256, 'global_every': 4, 'query_chunk': 128, 'dropout': 0.0, 'activation_checkpointing': False, 'loss_chunk': 256}
model4 119,399,680 {'vocab_size': 32768, 'width': 640, 'layers': 16, 'kernel_size': 3, 'dilation_cycle': 10, 'context': 2048, 'dropout': 0.0, 'activation_checkpointing': False, 'loss_chunk': 256}


## 5. Reusable token cache

This cell tokenizes each retained case once per distinct tokenizer. It does not retrain the tokenizer. Full document tokens are stored on disk; experiments select windows from these tokens without retokenizing.

The cache saves progress every 100 cases. After interruption, rerun the cell; only the unfinished part is repeated. A completed cache from a saved Kaggle output can be attached through `PREPARED_CACHE`. Save the `token_cache`, `assets`, and notebook after CPU preparation.


In [16]:
def object_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

def prepare_tokens(model_name, split):
    base = BASES[model_name]
    folder_name = base['tokenizer_sha256'][:16]
    wanted = dict(version=VERSION, tokenizer=base['tokenizer_sha256'],
                  index=object_hash(CASE_INDEX[split]),
                  source=digest_file(DATA_ROOT / ('Test/test.json' if split == 'test' else f'Train/{split}.json')))
    candidates = []
    if PREPARED_CACHE:
        candidates.append(Path(PREPARED_CACHE) / folder_name / split)
    destination = OUTPUT_ROOT / 'token_cache' / folder_name / split
    candidates.append(destination)
    for folder in candidates:
        if (folder / 'manifest.json').is_file():
            manifest = read_json(folder / 'manifest.json')
            if manifest['signature'] != wanted:
                raise ValueError(f'Token cache differs from current data/tokenizer: {folder}')
            if digest_file(folder / 'tokens.bin') != manifest['sha256']:
                raise ValueError(f'Token cache hash mismatch: {folder}')
            rows = read_json(folder / 'index.json')
            if len(rows) != len(CASE_INDEX[split]): raise ValueError('Incomplete cache index')
            return folder
    destination.mkdir(parents=True, exist_ok=True)
    partial = destination / 'progress.json'
    state = read_json(partial) if partial.exists() else dict(signature=wanted, rows=[], tokens=0)
    if state['signature'] != wanted: raise ValueError(f'Partial cache settings changed: {destination}')
    rows = state['rows']
    kept = {r['id'] for r in CASE_INDEX[split]}
    path = DATA_ROOT / ('Test/test.json' if split == 'test' else f'Train/{split}.json')
    binary = destination / 'tokens.bin'
    if state['tokens'] and (not binary.exists() or binary.stat().st_size < state['tokens'] * 2):
        raise ValueError('Partial token cache is shorter than its progress record')
    with open(binary, 'r+b' if binary.exists() else 'w+b') as output:
        output.truncate(state['tokens'] * 2)
        output.seek(0, 2)
        selected_position = 0
        for record in tqdm(iter_records(path), total=None, desc=f'Tokenize {split}'):
            identifier = record['id']
            if identifier not in kept: continue
            if selected_position < len(state['rows']):
                if rows[selected_position]['id'] != identifier: raise ValueError('Cache order changed')
                selected_position += 1
                continue
            ids = base['tokenizer'].encode(record['text']).ids
            if not ids or min(ids) < 0 or max(ids) >= min(65536, base['config']['vocab_size']):
                raise ValueError(f'Invalid token IDs for {identifier}')
            start = state['tokens']
            np.asarray(ids, dtype='<u2').tofile(output)
            state['tokens'] += len(ids)
            rows.append(dict(id=identifier, start=start, length=len(ids)))
            selected_position += 1
            if len(rows) % 100 == 0:
                output.flush()
                os.fsync(output.fileno())
                write_json(partial, state)
        output.flush()
        os.fsync(output.fileno())
    if [r['id'] for r in rows] != [r['id'] for r in CASE_INDEX[split]]:
        raise ValueError('Token cache does not match retained case order')
    write_json(destination / 'index.json', rows)
    lengths = [r['length'] for r in rows]
    write_json(destination / 'manifest.json', dict(signature=wanted, tokens=state['tokens'],
        sha256=digest_file(binary), token_percentiles=np.percentile(lengths,[50,90,95,99,100]).tolist()))
    return destination

TOKEN_FOLDERS = {}
def prepare_all_tokens():
    shared = {}
    for model_name in MODEL_NAMES:
        for split in ('train','dev','test'):
            key = (BASES[model_name]['tokenizer_sha256'], split)
            if key not in shared: shared[key] = prepare_tokens(model_name, split)
            TOKEN_FOLDERS[(model_name,split)] = shared[key]
    for key, folder in TOKEN_FOLDERS.items():
        print(key, read_json(folder / 'manifest.json')['token_percentiles'])


In [17]:
prepare_all_tokens()
print('Token percentiles above: 50, 90, 95, 99, 100. Cache:', OUTPUT_ROOT / 'token_cache')


('model1', 'train') [2473.0, 7822.0, 11023.199999999983, 23443.879999999997, 499429.0]
('model1', 'dev') [3883.0, 9491.0, 12833.5, 24404.600000000002, 60530.0]
('model1', 'test') [3927.0, 9626.600000000002, 13451.200000000012, 24350.87999999998, 56171.0]
('model2', 'train') [2473.0, 7822.0, 11023.199999999983, 23443.879999999997, 499429.0]
('model2', 'dev') [3883.0, 9491.0, 12833.5, 24404.600000000002, 60530.0]
('model2', 'test') [3927.0, 9626.600000000002, 13451.200000000012, 24350.87999999998, 56171.0]
('model3', 'train') [2473.0, 7822.0, 11023.199999999983, 23443.879999999997, 499429.0]
('model3', 'dev') [3883.0, 9491.0, 12833.5, 24404.600000000002, 60530.0]
('model3', 'test') [3927.0, 9626.600000000002, 13451.200000000012, 24350.87999999998, 56171.0]
('model4', 'train') [2473.0, 7822.0, 11023.199999999983, 23443.879999999997, 499429.0]
('model4', 'dev') [3883.0, 9491.0, 12833.5, 24404.600000000002, 60530.0]
('model4', 'test') [3927.0, 9626.600000000002, 13451.200000000012, 24350.87

## 6. Window selection and case batches

`tail` selects the last window. `head_tail` uses the first and last windows, unless the document fits in one. `hierarchical` selects up to `max_chunks` windows in document order, always including the first and last; long documents use evenly distributed windows. The coverage report shows how much text is retained. This bounded version does not claim to process every token in every case.

Chunks overlap where their selected positions permit it. Windows contain ordinary document tokens. Right padding is excluded from pooling. One batch element is one complete case representation, and the training loss is computed once per case.


In [18]:
def window_starts(length, cfg):
    size = cfg['chunk_tokens']
    if size < 2 or not 0 <= cfg['overlap'] < size: raise ValueError('Invalid chunk/overlap')
    last = max(0, length-size)
    if cfg['representation'] == 'tail': return [last]
    if cfg['representation'] == 'head_tail': return sorted(set([0,last]))
    if cfg['representation'] != 'hierarchical': raise ValueError('Unknown representation')
    if cfg['max_chunks'] < 2: raise ValueError('Hierarchical selection needs at least two chunks')
    starts = list(range(0,last+1,size-cfg['overlap']))
    if starts[-1] != last: starts.append(last)
    if len(starts) > cfg['max_chunks']:
        selected = np.linspace(0,len(starts)-1,cfg['max_chunks']).round().astype(int)
        starts = [starts[i] for i in selected]
    return starts

def max_case_chunks(cfg):
    return {'tail':1,'head_tail':2,'hierarchical':cfg['max_chunks']}[cfg['representation']]

class CaseDataset(Dataset):
    def __init__(self, model_name, split, cfg):
        folder = TOKEN_FOLDERS[(model_name,split)]
        self.tokens = np.memmap(folder/'tokens.bin',dtype='<u2',mode='r')
        self.rows = read_json(folder/'index.json')
        self.records = CASE_INDEX[split]
        self.cfg = cfg
        if [r['id'] for r in self.rows] != [r['id'] for r in self.records]:
            raise ValueError('Dataset/cache order mismatch')
    def __len__(self): return len(self.rows)
    def __getitem__(self,index):
        row = self.rows[index]
        starts = window_starts(row['length'],self.cfg)
        windows = [np.array(self.tokens[row['start']+s:row['start']+min(row['length'],s+self.cfg['chunk_tokens'])],dtype=np.int64) for s in starts]
        positions = [s/max(1,row['length']-1) for s in starts]
        label = self.records[index]['label']
        return windows, positions, (-1 if label is None else label), index

def collate_cases(samples):
    count = len(samples)
    chunks = max(len(s[0]) for s in samples)
    length = max(len(w) for s in samples for w in s[0])
    ids = torch.full((count,chunks,length),PAD,dtype=torch.long)
    mask = torch.zeros_like(ids,dtype=torch.bool)
    positions = torch.zeros(count,chunks,dtype=torch.float32)
    for i,(windows,places,_,_) in enumerate(samples):
        for j,(window,place) in enumerate(zip(windows,places)):
            ids[i,j,:len(window)] = torch.as_tensor(window)
            mask[i,j,:len(window)] = True
            positions[i,j] = place
    return dict(ids=ids, mask=mask, positions=positions,
                labels=torch.tensor([s[2] for s in samples]), indices=torch.tensor([s[3] for s in samples]))

def coverage_report(model_name,cfg):
    results = {}
    for split in ('train','dev','test'):
        rows = read_json(TOKEN_FOLDERS[(model_name,split)]/'index.json')
        covered,total,counts = 0,0,[]
        for row in rows:
            starts = window_starts(row['length'],cfg)
            end = 0
            for s in starts:
                stop = min(row['length'],s+cfg['chunk_tokens'])
                covered += max(0,stop-max(s,end)); end = max(end,stop)
            total += row['length']; counts.append(len(starts))
        results[split] = dict(token_coverage=covered/total,mean_chunks=float(np.mean(counts)),max_chunks=max(counts))
    print(json.dumps(results,indent=2))
    return results


## 7. Classification and adaptation

Each chunk combines its masked mean state with its last valid state. Model 2 instead combines the encoder mean with a one-token decoder query. Set `model2_pool='encoder_only'` for a separate encoder-only comparison; its unused decoder is frozen.

Chunks are combined by learned attention, or a bidirectional GRU followed by attention. Normalized document position is available to the aggregator. No language-model vocabulary logits are computed during classification.

LoRMA uses the identity-inflated multiplicative update from the authors' implementation. AdaLoRA uses PEFT. Both leave the document aggregator and classifier trainable. The smaller CNN student retains alternate Phase 1 blocks and their existing convolution dilations.


In [19]:
def set_activation_checkpointing(backbone,enabled):
    if hasattr(backbone,'gradient_checkpointing'): backbone.gradient_checkpointing=enabled
    if 'activation_checkpointing' in backbone.config: backbone.config['activation_checkpointing']=enabled

class LoRMAPlus(nn.Module):
    def __init__(self,base,rank,alpha):
        super().__init__()
        self.base=base
        self.base.requires_grad_(False)
        self.A=nn.Parameter(base.weight.new_empty(rank,base.out_features))
        self.B=nn.Parameter(base.weight.new_zeros(base.out_features,rank))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))
        self.scale=alpha/rank
    def forward(self,x):
        z=F.linear(x,self.base.weight,None)
        z=z+self.scale*F.linear(F.linear(z,self.A),self.B)
        return z if self.base.bias is None else z+self.base.bias

def install_lorma(module,rank,alpha,prefix=''):
    names=[]
    for name,child in list(module.named_children()):
        path=f'{prefix}.{name}' if prefix else name
        if isinstance(child,nn.Linear):
            setattr(module,name,LoRMAPlus(child,rank,alpha)); names.append(path)
        else: names.extend(install_lorma(child,rank,alpha,path))
    return names

def decoder_hidden(backbone,ids):
    if ids.shape[1]>backbone.config['context']: raise ValueError('Context exceeded')
    pos=torch.arange(ids.shape[1],device=ids.device)
    x=backbone.drop(backbone.token(ids)+backbone.position(pos))
    for block in backbone.blocks:
        if backbone.training and backbone.gradient_checkpointing:
            x=checkpoint(lambda z,layer=block: layer(z)[0],x,use_reentrant=False)
        else: x=block(x)[0]
    return backbone.final_norm(x)

class ChunkEncoder(nn.Module):
    def __init__(self,model_name,cfg):
        super().__init__()
        base=BASES[model_name]
        self.model_name=model_name
        self.pool=cfg['model2_pool']
        self.config={'model_type':'custom_cjpe'}
        self.backbone=MODEL_CLASSES[model_name](**copy.deepcopy(base['config']))
        self.backbone.load_state_dict(base['state'],strict=True)
        set_activation_checkpointing(self.backbone,cfg['activation_checkpointing'])
        if cfg['chunk_tokens']>base['config'].get('context',1408):
            raise ValueError('Chunk length exceeds the configured/pretrained input limit')
        if cfg['student_keep'] is not None:
            if model_name!='model4': raise ValueError('Block reduction is implemented for Model 4')
            keep=cfg['student_keep']
            if not keep or sorted(set(keep))!=keep or max(keep)>=len(self.backbone.blocks):
                raise ValueError('Invalid retained student blocks')
            self.backbone.blocks=nn.ModuleList([self.backbone.blocks[i] for i in keep])
            self.backbone.dilations=[self.backbone.dilations[i] for i in keep]
            self.backbone.config['layers']=len(keep)
        if model_name=='model2' and self.pool=='encoder_only':
            self.backbone.decoder.requires_grad_(False)
            self.backbone.dec_norm.requires_grad_(False)
    def forward(self,ids,mask):
        if self.model_name=='model2':
            hidden,valid=self.backbone.encode(ids)
            if self.pool=='encoder_query':
                query=torch.full((len(ids),1),BOS,dtype=torch.long,device=ids.device)
                decoded,_=self.backbone.decode(query,hidden,valid)
                summary=decoded[:,-1].float()
            elif self.pool=='encoder_only':
                summary=(hidden.float()*mask.unsqueeze(-1)).sum(1)/mask.sum(1,keepdim=True).clamp_min(1)
            else: raise ValueError(self.pool)
        else:
            hidden=decoder_hidden(self.backbone,ids) if self.model_name=='model1' else self.backbone.hidden(ids)[0]
            last=mask.long().sum(1)-1
            summary=hidden[torch.arange(len(ids),device=ids.device),last].float()
        mean=(hidden.float()*mask.unsqueeze(-1)).sum(1)/mask.sum(1,keepdim=True).clamp_min(1)
        return torch.cat([mean,summary],dim=-1)

class CaseClassifier(nn.Module):
    def __init__(self,model_name,method,cfg,total_steps):
        super().__init__()
        self.model_name=model_name; self.method=method; self.cfg=copy.deepcopy(cfg)
        self.encoder=ChunkEncoder(model_name,cfg)
        self.adapted=[]
        if method=='lorma':
            self.encoder.requires_grad_(False)
            self.adapted=install_lorma(self.encoder.backbone.blocks,cfg['rank'],cfg['alpha'])
        elif method=='adalora':
            from peft import AdaLoraConfig
            from peft.tuners.adalora import AdaLoraModel
            targets=[n for n,m in self.encoder.named_modules() if isinstance(m,nn.Linear)]
            adaptation=AdaLoraConfig(target_modules=targets,init_r=cfg['initial_rank'],
                target_r=cfg['rank'],lora_alpha=cfg['alpha'],lora_dropout=0.0,
                total_step=total_steps,tinit=max(1,int(total_steps*.1)),
                tfinal=max(1,int(total_steps*.2)),deltaT=10,orth_reg_weight=cfg['orth_reg_weight'])
            self.encoder=AdaLoraModel(self.encoder,adaptation,'default')
            self.adapted=targets
        elif method not in ('full','kd','control'):
            raise ValueError(method)
        width=BASES[model_name]['config']['width']
        dim=cfg['head_dim']
        self.projection=nn.Sequential(nn.LayerNorm(2*width),nn.Dropout(cfg['dropout']),
                                      nn.Linear(2*width,dim),nn.GELU())
        self.position=nn.Linear(1,dim,bias=False)
        self.gru=nn.GRU(dim,dim//2,batch_first=True,bidirectional=True) if cfg['aggregator']=='bigru' else None
        if cfg['aggregator'] not in ('attention','bigru'): raise ValueError('Unknown aggregator')
        self.attention=nn.Sequential(nn.Linear(dim,dim//2),nn.Tanh(),nn.Linear(dim//2,1))
        self.classifier=nn.Sequential(nn.LayerNorm(dim),nn.Dropout(cfg['dropout']),nn.Linear(dim,2))
    def forward(self,ids,mask,positions):
        b,k,l=ids.shape
        valid=mask.any(-1)
        flat_valid=valid.flatten()
        features=self.encoder(ids.reshape(b*k,l)[flat_valid],mask.reshape(b*k,l)[flat_valid])
        features=self.projection(features)
        dense=features.new_zeros(b*k,features.shape[-1])
        dense[flat_valid]=features
        dense=dense.reshape(b,k,-1)+self.position(positions.unsqueeze(-1).to(features.dtype))
        if self.gru is not None:
            packed=pack_padded_sequence(dense,valid.sum(1).cpu(),batch_first=True,enforce_sorted=False)
            packed,_=self.gru(packed)
            dense,_=pad_packed_sequence(packed,batch_first=True,total_length=k)
        scores=self.attention(dense).squeeze(-1).float().masked_fill(~valid,float('-inf'))
        weights=scores.softmax(-1).to(dense.dtype)
        pooled=(dense*weights.unsqueeze(-1)).sum(1)
        return self.classifier(pooled).float()

def parameter_report(model):
    return dict(total=sum(p.numel() for p in model.parameters()),
                trainable=sum(p.numel() for p in model.parameters() if p.requires_grad),
                adapted_modules=len(model.adapted))

def make_model(model_name,method,cfg,total_steps,device):
    return CaseClassifier(model_name,method,cfg,total_steps).to(device)

def adalora_penalty(model):
    if model.method!='adalora': return next(model.parameters()).new_zeros(())
    penalties=[]
    for name,p in model.encoder.named_parameters():
        if 'lora_A' in name or 'lora_B' in name:
            w=p.float()
            gram=w@w.T if 'lora_A' in name else w.T@w
            penalties.append(torch.linalg.matrix_norm(gram-torch.eye(len(gram),device=w.device)))
    return torch.stack(penalties).mean()*model.cfg['orth_reg_weight']

def adapter_state(model):
    if model.method!='adalora': return None
    allocator=model.encoder.rankallocator
    return dict(buffers={n:{k:v.detach().cpu() for k,v in getattr(allocator,n).items()}
                         for n in ('ipt','exp_avg_ipt','exp_avg_unc')},
                rank_pattern=copy.deepcopy(model.encoder.peft_config['default'].rank_pattern))

def restore_adapter_state(model,state,device):
    if state is None: return
    for name,values in state['buffers'].items():
        setattr(model.encoder.rankallocator,name,{k:v.to(device) for k,v in values.items()})
    model.encoder.peft_config['default'].rank_pattern=state['rank_pattern']


## 8. Backbone and padding checks

These checks use temporary models. They verify tokenizer round trips, finite classification scores, right-padding invariance and classifier gradients. The original checkpoints are loaded strictly. Hardware stress tests are separate and use maximum configured case lengths.


In [20]:
def check_backbones():
    device='cuda' if torch.cuda.is_available() else 'cpu'
    for name in MODEL_NAMES:
        tok=BASES[name]['tokenizer']
        for text in ['Section 42: भारत ₹100.','a  b\n c','<s> literal text']:
            assert tok.decode(tok.encode(text).ids)==text
        cfg=dict(DEFAULT,chunk_tokens=64,representation='tail',dropout=0.0)
        model=make_model(name,'full',cfg,100,device)
        tokens=tok.encode('The court considered the appeal and the evidence.').ids[:48]
        batch=collate_cases([(np.asarray(tokens)[None,:], [0.0], 1, 0)])
        x={k:v.to(device) for k,v in batch.items() if k in ('ids','mask','positions')}
        model.eval()
        with torch.no_grad():
            original=model(**x)
            padded=dict(x,ids=F.pad(x['ids'],(0,7),value=PAD),mask=F.pad(x['mask'],(0,7),value=False))
            other=model(**padded)
        torch.testing.assert_close(original,other,rtol=2e-4,atol=2e-4)
        model.train(); F.cross_entropy(model(**x),torch.tensor([1],device=device)).backward()
        assert any(p.grad is not None and p.grad.abs().sum()>0 for p in model.classifier.parameters())
        assert all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None)
        print(name,'passed',parameter_report(model),flush=True)
        del model, x, original, other, padded
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

check_backbones()


model1 passed {'total': 120008707, 'trainable': 120008707, 'adapted_modules': 0}
model2 passed {'total': 122548739, 'trainable': 122548739, 'adapted_modules': 0}
model3 passed {'total': 118566915, 'trainable': 118566915, 'adapted_modules': 0}
model4 passed {'total': 119575171, 'trainable': 119575171, 'adapted_modules': 0}


## 9. Training helpers

The capacity check tries 1, 2 and 4 cases per GPU, with and without activation checkpointing, while keeping the effective batch fixed. Each candidate performs full optimizer updates on maximum-size case batches, including all selected chunks. It measures reserved memory and chooses the fastest passing candidate with the requested headroom. It does not alter a run checkpoint.

On two GPUs, cases are split across the GPUs with PyTorch DataParallel. Cases remain intact for document aggregation. Reported speed includes accumulation; epoch estimates exclude validation and file saving. Preparation/tokenization is CPU work.


In [21]:
def sync_cuda():
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()): torch.cuda.synchronize(i)

def autocast_context(device):
    return torch.autocast('cuda',dtype=torch.float16) if str(device).startswith('cuda') else contextlib.nullcontext()

def parallel_model(model):
    return nn.DataParallel(model) if torch.cuda.device_count()>1 and next(model.parameters()).is_cuda else model

def model_inputs(batch,device):
    return {k:batch[k].to(device) for k in ('ids','mask','positions')}

def make_optimizer(model,cfg):
    groups=[]
    for is_backbone in (True,False):
        for decay in (True,False):
            params=[p for n,p in model.named_parameters() if p.requires_grad
                    and n.startswith('encoder.')==is_backbone and (p.ndim>1)==decay]
            if params:
                lr=(cfg['adapter_lr'] if model.method in ('lorma','adalora') else cfg['backbone_lr']) if is_backbone else cfg['head_lr']
                groups.append(dict(params=params,lr=lr,weight_decay=cfg['weight_decay'] if decay else 0.0))
    return torch.optim.AdamW(groups)

def make_scheduler(optimizer,cfg,total):
    warmup=max(1,int(total*cfg['warmup_fraction']))
    def factor(step):
        if step<warmup: return (step+1)/warmup
        progress=min(1.0,(step-warmup)/max(1,total-warmup))
        return cfg['min_lr_ratio']+(1-cfg['min_lr_ratio'])*0.5*(1+math.cos(math.pi*progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer,factor)

def save_torch(path,payload):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.tmp')
    torch.save(payload,tmp); tmp.replace(path)

def rng_state():
    return dict(python=random.getstate(),numpy=np.random.get_state(),torch=torch.get_rng_state(),
                cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [])

def restore_rng(state):
    random.setstate(state['python']); np.random.set_state(state['numpy']); torch.set_rng_state(state['torch'])
    if torch.cuda.is_available() and state['cuda']:
        if len(state['cuda'])!=torch.cuda.device_count():
            raise ValueError('Resume on the same number of GPUs for reproducible continuation')
        torch.cuda.set_rng_state_all(state['cuda'])

def metric_report(labels,probabilities,threshold=0.5):
    labels=np.asarray(labels,dtype=np.int64)
    prediction=(np.asarray(probabilities)>=threshold).astype(np.int64)
    return dict(macro_f1=float(f1_score(labels,prediction,average='macro',labels=[0,1],zero_division=0)),
                precision=float(precision_score(labels,prediction,zero_division=0)),
                recall=float(recall_score(labels,prediction,zero_division=0)),
                confusion_matrix=confusion_matrix(labels,prediction,labels=[0,1]).tolist(),
                predicted_counts=np.bincount(prediction,minlength=2).tolist())

@torch.no_grad()
def predict_logits(model,dataset,batch_size,device,description='Evaluate'):
    model.eval(); inference=parallel_model(model)
    output=[]
    loader=DataLoader(dataset,batch_size=batch_size,shuffle=False,collate_fn=collate_cases,
                      num_workers=0,generator=torch.Generator().manual_seed(0))
    for batch in tqdm(loader,desc=description,leave=False):
        with autocast_context(device): logits=inference(**model_inputs(batch,device))
        if not torch.isfinite(logits).all(): raise FloatingPointError('Non-finite inference logits')
        output.append(logits.float().cpu())
    return torch.cat(output)

EXPERIMENTS={}
def register(name,model_name,method,**overrides):
    if not re.fullmatch(r'[a-z0-9_]+',name): raise ValueError('Use a simple unique run name')
    cfg=dict(DEFAULT,**overrides)
    if cfg['effective_batch']<1 or cfg['epochs']<1: raise ValueError('Invalid training budget')
    EXPERIMENTS[name]=dict(name=name,model=model_name,method=method,config=cfg)
    return name

def run_folder(name): return OUTPUT_ROOT/'runs'/name

def total_updates(cfg): return math.ceil(len(CASE_INDEX['train'])/cfg['effective_batch'])*cfg['epochs']

def hardware_signature(spec):
    return dict(version=VERSION,spec=spec,checkpoint=BASES[spec['model']]['checkpoint_sha256'],
                devices=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
                torch=torch.__version__,train_examples=len(CASE_INDEX['train']))

def capacity_check(name):
    spec=EXPERIMENTS[name]; cfg=spec['config']; folder=run_folder(name)
    if not torch.cuda.is_available(): raise RuntimeError('Enable a Kaggle GPU before the capacity check')
    path=folder/'hardware.json'; signature=hardware_signature(spec)
    if path.exists():
        saved=read_json(path)
        if saved['signature']!=signature: raise ValueError('Hardware/settings changed; use a new run name')
        print('Saved capacity selection:',saved['selected']); return saved['selected']
    results=[]; n_gpu=torch.cuda.device_count()
    for checkpointing in (True,False):
        for micro in (1,2,4):
            batch_size=micro*n_gpu
            if batch_size>cfg['effective_batch'] or cfg['effective_batch']%batch_size: continue
            trial=dict(cfg,micro_batch_per_gpu=micro,activation_checkpointing=checkpointing)
            model=net=optimizer=scaler=None
            print(f'Testing {micro} cases/GPU; checkpointing={checkpointing}',flush=True)
            try:
                gc.collect(); torch.cuda.empty_cache(); seed_everything(cfg['seed'])
                model=make_model(spec['model'],spec['method'],trial,total_updates(cfg),'cuda')
                net=parallel_model(model); model.train(); optimizer=make_optimizer(model,trial)
                scaler=torch.amp.GradScaler('cuda')
                k=max_case_chunks(cfg); l=cfg['chunk_tokens']
                x=dict(ids=torch.randint(4,BASES[spec['model']]['config']['vocab_size'],(batch_size,k,l),device='cuda'),
                       mask=torch.ones(batch_size,k,l,dtype=torch.bool,device='cuda'),
                       positions=torch.linspace(0,1,k,device='cuda')[None,:].expand(batch_size,-1))
                y=torch.arange(batch_size,device='cuda')%2
                for i in range(n_gpu): torch.cuda.reset_peak_memory_stats(i)
                durations=[]
                accumulation=cfg['effective_batch']//batch_size
                for repeat in range(4):
                    sync_cuda(); start=time.perf_counter(); optimizer.zero_grad(set_to_none=True)
                    for _ in range(accumulation):
                        with autocast_context('cuda'):
                            loss=F.cross_entropy(net(**x),y)/accumulation
                        scaler.scale(loss).backward()
                    if spec['method']=='adalora': scaler.scale(adalora_penalty(model)).backward()
                    scaler.unscale_(optimizer)
                    norm=nn.utils.clip_grad_norm_(model.parameters(),cfg['clip'])
                    if not torch.isfinite(norm): raise FloatingPointError('Non-finite stress-test gradients')
                    scaler.step(optimizer); scaler.update()
                    if spec['method']=='adalora': model.encoder.update_and_allocate(repeat+1)
                    sync_cuda(); durations.append(time.perf_counter()-start)
                memory=[]
                for i in range(n_gpu):
                    free,total=torch.cuda.mem_get_info(i)
                    reserved=torch.cuda.max_memory_reserved(i)
                    current=torch.cuda.memory_reserved(i)
                    headroom=min(free,total-reserved)
                    memory.append(dict(gpu=i,peak_reserved_gib=reserved/2**30,
                                       peak_tensor_gib=torch.cuda.max_memory_allocated(i)/2**30,
                                       headroom_gib=headroom/2**30))
                row=dict(micro_batch_per_gpu=micro,activation_checkpointing=checkpointing,
                         seconds_per_update=float(np.median(durations[1:])),memory=memory,
                         passed=all(m['headroom_gib']>=cfg['capacity_headroom_gib'] for m in memory))
            except torch.cuda.OutOfMemoryError:
                row=dict(micro_batch_per_gpu=micro,activation_checkpointing=checkpointing,passed=False,error='CUDA out of memory')
            finally:
                model=net=optimizer=scaler=None
                if 'loss' in locals(): del loss
                if 'x' in locals(): del x,y
                gc.collect(); torch.cuda.empty_cache()
            results.append(row); print(row,flush=True)
    passing=[r for r in results if r['passed']]
    if not passing: raise RuntimeError('No capacity candidate passed. Reduce chunk length/count in a new run.')
    selected=min(passing,key=lambda r:r['seconds_per_update'])
    write_json(path,dict(signature=signature,results=results,selected=selected))
    print('Selected:',selected)
    print('Training-only epoch estimate (minutes):',selected['seconds_per_update']*math.ceil(len(CASE_INDEX['train'])/cfg['effective_batch'])/60)
    return selected

def resolved_spec(name):
    spec=copy.deepcopy(EXPERIMENTS[name]); path=run_folder(name)/'hardware.json'
    if not path.is_file(): raise RuntimeError(f'Run capacity_check({name!r}) first')
    hardware=read_json(path)
    if hardware['signature']!=hardware_signature(spec): raise ValueError('Capacity settings mismatch')
    for key in ('micro_batch_per_gpu','activation_checkpointing'):
        spec['config'][key]=hardware['selected'][key]
    return spec

def training_signature(spec,teacher):
    return dict(version=VERSION,spec=spec,base_checkpoint=BASES[spec['model']]['checkpoint_sha256'],
                tokenizer=BASES[spec['model']]['tokenizer_sha256'],
                cases={split:object_hash(CASE_INDEX[split]) for split in ('train','dev')},
                tokens={split:digest_file(TOKEN_FOLDERS[(spec['model'],split)]/'tokens.bin') for split in ('train','dev')},
                teacher=teacher,
                libraries={p:importlib.metadata.version(p) for p in ['peft','transformers']})

def log_metric(folder,row):
    with open(folder/'metrics.jsonl','a',encoding='utf-8') as f: f.write(json.dumps(row)+'\n')
    print(row,flush=True)

def train_run(name,additional_steps=None,teacher_name=None):
    spec=resolved_spec(name); cfg=spec['config']; folder=run_folder(name)
    device='cuda' if torch.cuda.is_available() else 'cpu'
    teacher_reference=None; teacher_logits=None
    if spec['method']=='kd':
        if teacher_name is None: raise ValueError('Select a completed teacher first')
        teacher_logits,teacher_reference=cache_teacher(teacher_name)
    signature=training_signature(spec,teacher_reference)
    if (folder/'signature.json').exists() and read_json(folder/'signature.json')!=signature:
        raise ValueError('Run inputs/settings changed. Register a new run name.')
    if additional_steps is not None and additional_steps<1: raise ValueError('additional_steps must be positive')
    write_json(folder/'signature.json',signature)
    write_json(folder/'config.json',spec)
    seed_everything(cfg['seed'])
    train_data=CaseDataset(spec['model'],'train',cfg); dev_data=CaseDataset(spec['model'],'dev',cfg)
    total=total_updates(cfg)
    model=make_model(spec['model'],spec['method'],cfg,total,device)
    parameters=parameter_report(model); print('Parameters:',parameters,flush=True)
    if teacher_reference and parameters['total']>=teacher_reference['parameters']:
        raise ValueError('Distillation student must be smaller than the teacher')
    net=parallel_model(model); optimizer=make_optimizer(model,cfg)
    scheduler=make_scheduler(optimizer,cfg,total)
    scaler=torch.amp.GradScaler('cuda',enabled=device=='cuda')
    state=dict(epoch=0,position=0,step=0,best_macro_f1=-1.0,bad_epochs=0,complete=False,skipped=0)
    latest=folder/'latest.pt'
    if latest.exists():
        saved=torch.load(latest,map_location='cpu',weights_only=False)
        if saved['signature']!=signature: raise ValueError('Resume signature mismatch')
        model.load_state_dict(saved['model'],strict=True); optimizer.load_state_dict(saved['optimizer'])
        scheduler.load_state_dict(saved['scheduler']); scaler.load_state_dict(saved['scaler'])
        restore_adapter_state(model,saved['adapter'],device)
        state=saved['progress']; restore_rng(saved['rng']); del saved
        print('Resumed:',state,flush=True)
    start_step=state['step']; started=time.monotonic()
    batch_size=cfg['micro_batch_per_gpu']*max(1,torch.cuda.device_count())
    class_weights=None
    if cfg['class_weighting']:
        counts=np.bincount([r['label'] for r in CASE_INDEX['train']],minlength=2)
        class_weights=torch.tensor(len(train_data)/(2*counts),dtype=torch.float32,device=device)

    def save_latest():
        save_torch(latest,dict(signature=signature,model={k:v.detach().cpu() for k,v in model.state_dict().items()},
            optimizer=optimizer.state_dict(),scheduler=scheduler.state_dict(),scaler=scaler.state_dict(),
            adapter=adapter_state(model),progress=copy.deepcopy(state),rng=rng_state()))
        write_json(folder/'progress.json',state)

    def validate(count_patience):
        logits=predict_logits(model,dev_data,batch_size,device)
        probabilities=logits.softmax(-1)[:,1].numpy()
        metrics=metric_report([r['label'] for r in CASE_INDEX['dev']],probabilities)
        log_metric(folder,dict(step=state['step'],epoch=state['epoch'],validation_macro_f1=metrics['macro_f1']))
        if metrics['macro_f1']>state['best_macro_f1']+1e-8:
            state['best_macro_f1']=metrics['macro_f1']; state['bad_epochs']=0
            save_torch(folder/'best.pt',dict(signature=signature,model={k:v.detach().cpu() for k,v in model.state_dict().items()},
                step=state['step'],epoch=state['epoch'],metrics=metrics,parameters=parameters))
            write_json(folder/'best_dev_probabilities.json',[dict(id=r['id'],gold=r['label'],probability=float(p))
                       for r,p in zip(CASE_INDEX['dev'],probabilities)])
        elif count_patience: state['bad_epochs']+=1
        model.train()

    try:
        if state['step']==0 and not (folder/'best.pt').exists(): validate(False); save_latest()
        while not state['complete'] and state['epoch']<cfg['epochs']:
            order=np.random.default_rng(cfg['seed']+state['epoch']).permutation(len(train_data))
            model.train()
            while state['position']<len(order):
                indices=order[state['position']:state['position']+cfg['effective_batch']]
                optimizer.zero_grad(set_to_none=True); loss_sum=0.0
                for begin in range(0,len(indices),batch_size):
                    selected=indices[begin:begin+batch_size]
                    batch=collate_cases([train_data[int(i)] for i in selected])
                    targets=batch['labels'].to(device)
                    with autocast_context(device):
                        logits=net(**model_inputs(batch,device))
                        ce=F.cross_entropy(logits,targets,weight=class_weights,
                            label_smoothing=cfg['label_smoothing'],reduction='none')
                        losses=ce
                        if teacher_logits is not None:
                            t=cfg['kd_temperature']
                            soft=teacher_logits[selected].to(device).softmax(-1) if t==1 else (teacher_logits[selected].to(device)/t).softmax(-1)
                            kl=F.kl_div(F.log_softmax(logits.float()/t,dim=-1),soft,reduction='none').sum(-1)*t*t
                            losses=(1-cfg['kd_alpha'])*ce+cfg['kd_alpha']*kl
                        loss=losses.sum()/len(indices)
                    if not torch.isfinite(loss): raise FloatingPointError('Non-finite loss; latest saved checkpoint remains available')
                    scaler.scale(loss).backward(); loss_sum+=float(loss.detach())
                if spec['method']=='adalora': scaler.scale(adalora_penalty(model)).backward()
                scaler.unscale_(optimizer)
                norm=nn.utils.clip_grad_norm_(model.parameters(),cfg['clip'])
                before=scaler.get_scale(); scaler.step(optimizer); scaler.update()
                skipped=scaler.get_scale()<before
                if not torch.isfinite(norm) and not skipped: raise FloatingPointError('Non-finite gradients')
                if skipped:
                    state['skipped']+=1
                    if state['skipped']>20: raise FloatingPointError('Too many skipped AMP updates')
                    print('AMP update skipped; retrying the same cases',flush=True)
                    continue
                state['step']+=1
                if spec['method']=='adalora': model.encoder.update_and_allocate(state['step'])
                scheduler.step()
                state['position']+=len(indices)
                if state['step']==start_step+1 or state['step']%cfg['log_every']==0:
                    log_metric(folder,dict(step=state['step'],epoch=state['epoch'],loss=loss_sum,
                        lr=optimizer.param_groups[0]['lr'],gradient_norm=float(norm),skipped=state['skipped'],
                        elapsed_minutes=(time.monotonic()-started)/60))
                if state['step']%cfg['save_every']==0: save_latest()
                pause=(additional_steps is not None and state['step']-start_step>=additional_steps) or (time.monotonic()-started)>=cfg['max_hours']*3600
                if pause and state['position']<len(order):
                    validate(False); save_latest()
                    return dict(name=name,**state,session_seconds=time.monotonic()-started)
            state['epoch']+=1; state['position']=0
            validate(True)
            state['complete']=state['epoch']>=cfg['epochs'] or state['bad_epochs']>=cfg['patience']
            if state['complete']: state['stop_reason']='epochs' if state['epoch']>=cfg['epochs'] else 'early_stopping'
            save_latest()
            if additional_steps is not None and state['step']-start_step>=additional_steps: break
            if time.monotonic()-started>=cfg['max_hours']*3600: break
        report=dict(name=name,**state,parameters=parameters,session_seconds=time.monotonic()-started)
        write_json(folder/'summary.json',report); print(report,flush=True)
        return report
    finally:
        optimizer.zero_grad(set_to_none=True)
        del net,model,optimizer,scheduler,scaler
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()


### Professor evaluator

The following cell writes the supplied evaluator verbatim to the assets folder. It is invoked with its original command-line arguments during validation.


In [23]:
EVALUATOR_PATH = ASSET_DIR / "evaluate_phase4.py"
EVALUATOR_SOURCE = r'''
#!/usr/bin/env python3
"""Local evaluator for Phase 4.

Truth JSON: a list of objects, [{"id":"...", "gold":0|1}, ...]
Prediction JSON: [{"id":"...", "pred":0|1}, ...]
"""
import argparse,json,sys
from sklearn.metrics import f1_score,precision_score,recall_score

def read_truth(path):
    try:
        x=json.load(open(path,encoding="utf-8"))
    except json.JSONDecodeError as e:
        raise ValueError(f"Truth must be a JSON file containing a list of records: {e}")
    if not isinstance(x,list) or not x: raise ValueError('Truth must be a non-empty JSON list.')
    out={}
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'gold' not in r: raise ValueError(f'Truth item {n} must contain id and gold.')
        if not isinstance(r['id'],str) or not r['id'].strip() or isinstance(r['gold'],bool) or r['gold'] not in (0,1): raise ValueError(f'Invalid truth item {n}.')
        if r['id'] in out: raise ValueError(f'Duplicate truth ID: {r["id"]}')
        out[r['id']]=r['gold']
    return out

def read_pred(path):
    x=json.load(open(path,encoding='utf-8'))
    if not isinstance(x,list) or not x: raise ValueError('Prediction must be a non-empty JSON list.')
    out={}
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'pred' not in r: raise ValueError(f'Prediction item {n} must contain id and pred.')
        if not isinstance(r['id'],str) or isinstance(r['pred'],bool) or r['pred'] not in (0,1): raise ValueError(f'Invalid prediction item {n}.')
        if r['id'] in out: raise ValueError(f'Duplicate prediction ID: {r["id"]}')
        out[r['id']]=r['pred']
    return out

def main():
    ap=argparse.ArgumentParser(description='Evaluate Phase 4 binary predictions on your own held-out split.')
    ap.add_argument('--truth',required=True); ap.add_argument('--prediction',required=True); a=ap.parse_args()
    t=read_truth(a.truth); p=read_pred(a.prediction)
    if set(t)!=set(p): raise ValueError(f"ID mismatch: missing={len(set(t)-set(p))}, extra={len(set(p)-set(t))}")
    yt=[t[i] for i in t]; yp=[p[i] for i in t]
    print(f'Examples: {len(t)}')
    print(f'Macro-F1: {f1_score(yt,yp,average="macro",labels=[0,1],zero_division=0):.6f}')
    print(f'Precision: {precision_score(yt,yp,average="binary",pos_label=1,zero_division=0):.6f}')
    print(f'Recall: {recall_score(yt,yp,average="binary",pos_label=1,zero_division=0):.6f}')
if __name__=='__main__':
    try: main()
    except Exception as e: print(f'EVALUATION FAILED: {e}',file=sys.stderr); sys.exit(1)
'''.lstrip("\n")
EVALUATOR_PATH.write_text(EVALUATOR_SOURCE, encoding="utf-8")
print("Evaluator:", EVALUATOR_PATH)


Evaluator: /kaggle/working/phase4_cjpe/assets/evaluate_phase4.py


## 10. Validation, teacher cache and submissions

The professor's evaluator reports Macro-F1 over both classes and binary precision/recall for class 1. The default decision threshold is 0.5. Optional coarse threshold selection uses development predictions only and writes a separate export folder. Keep threshold 0.5 as the reference result.

Test archives contain exactly `prediction.json` at the ZIP root. Every original test ID appears once, with an integer prediction. The validation cell saves probabilities, a confusion matrix, per-class scores and the professor's output.


In [24]:
def load_trained(name,device):
    folder=run_folder(name)
    spec=read_json(folder/'config.json'); saved=torch.load(folder/'best.pt',map_location='cpu',weights_only=False)
    if saved['signature']!=read_json(folder/'signature.json'): raise ValueError('Checkpoint signature mismatch')
    if saved['signature']['base_checkpoint']!=BASES[spec['model']]['checkpoint_sha256']:
        raise ValueError('Different Phase 1 weights attached')
    if saved['signature']['cases']['dev']!=object_hash(CASE_INDEX['dev']):
        raise ValueError('Development cases differ from the checkpoint run')
    model=make_model(spec['model'],spec['method'],spec['config'],total_updates(spec['config']),device)
    model.load_state_dict(saved['model'],strict=True)
    model.eval()
    return model,spec,saved

def validate_predictions(predictions,expected):
    if not isinstance(predictions,list) or len(predictions)!=len(expected): raise ValueError('Prediction count mismatch')
    ids=[]
    for row in predictions:
        if set(row)!={'id','pred'} or not isinstance(row['id'],str) or type(row['pred']) is not int or row['pred'] not in (0,1):
            raise ValueError(f'Invalid prediction: {row}')
        ids.append(row['id'])
    if len(set(ids))!=len(ids) or set(ids)!={r['id'] for r in expected}: raise ValueError('Prediction IDs mismatch')

def validate_run(name):
    folder=run_folder(name); device='cuda' if torch.cuda.is_available() else 'cpu'
    model,spec,saved=load_trained(name,device)
    expected_f1=saved['metrics']['macro_f1']
    try:
        cfg=spec['config']; data=CaseDataset(spec['model'],'dev',cfg)
        logits=predict_logits(model,data,cfg['micro_batch_per_gpu']*max(1,torch.cuda.device_count()),device)
        probabilities=logits.softmax(-1)[:,1].numpy()
    finally:
        del model,saved
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    truth=[r['label'] for r in CASE_INDEX['dev']]
    predictions=[dict(id=r['id'],pred=int(p>=0.5)) for r,p in zip(CASE_INDEX['dev'],probabilities)]
    validate_predictions(predictions,CASE_INDEX['dev'])
    out=folder/'validation'; out.mkdir(parents=True,exist_ok=True)
    write_json(out/'prediction.json',predictions)
    write_json(out/'probabilities.json',[dict(id=r['id'],gold=r['label'],probability=float(p)) for r,p in zip(CASE_INDEX['dev'],probabilities)])
    report=metric_report(truth,probabilities)
    report['saved_best_macro_f1']=expected_f1
    if abs(report['macro_f1']-expected_f1)>1e-6:
        print('Validation differs from the saved checkpoint score. Check numerical/device differences before submission.')
    report['per_class']=classification_report(truth,[r['pred'] for r in predictions],labels=[0,1],output_dict=True,zero_division=0)
    write_json(out/'metrics.json',report)
    result=subprocess.run([sys.executable,str(EVALUATOR_PATH),'--truth',str(ASSET_DIR/'dev_truth.json'),
                           '--prediction',str(out/'prediction.json')],capture_output=True,text=True)
    print(result.stdout,flush=True)
    if result.stderr: print(result.stderr)
    (out/'prof_evaluation.txt').write_text(result.stdout+'\n'+result.stderr,encoding='utf-8')
    write_json(out/'provenance.json',dict(checkpoint=digest_file(folder/'best.pt'),
               evaluator=digest_file(EVALUATOR_PATH),truth=digest_file(ASSET_DIR/'dev_truth.json'),
               predictions=digest_file(out/'prediction.json'),returncode=result.returncode))
    result.check_returncode()
    for label in ('0','1'): print(label,report['per_class'][label])
    return report

def threshold_comparison(name):
    folder=run_folder(name); path=folder/'validation/probabilities.json'
    if not path.exists(): validate_run(name)
    rows=read_json(path)
    results=[dict(threshold=t,**metric_report([r['gold'] for r in rows],[r['probability'] for r in rows],t))
             for t in [0.35,0.40,0.45,0.50,0.55,0.60,0.65]]
    best=max(results,key=lambda r:(r['macro_f1'],-abs(r['threshold']-.5)))
    write_json(folder/'validation/thresholds.json',dict(results=results,selected=best,
        checkpoint=digest_file(folder/'best.pt')))
    for r in results: print(r['threshold'],r['macro_f1'])
    return best['threshold']

def training_sample_report(name,per_class=64):
    device='cuda' if torch.cuda.is_available() else 'cpu'
    model,spec,saved=load_trained(name,device)
    selected=[]; rng=np.random.default_rng(42)
    for label in (0,1):
        indices=[i for i,r in enumerate(CASE_INDEX['train']) if r['label']==label]
        selected.extend(rng.choice(indices,size=min(per_class,len(indices)),replace=False).tolist())
    try:
        cfg=spec['config']
        subset=torch.utils.data.Subset(CaseDataset(spec['model'],'train',cfg),selected)
        logits=predict_logits(model,subset,cfg['micro_batch_per_gpu']*max(1,torch.cuda.device_count()),device,'Training sample')
    finally:
        del model,saved
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    gold=[CASE_INDEX['train'][i]['label'] for i in selected]
    report=metric_report(gold,logits.softmax(-1)[:,1].numpy())
    report['examples']=len(selected)
    report['case_ids']=[CASE_INDEX['train'][i]['id'] for i in selected]
    write_json(run_folder(name)/'validation/training_sample.json',report)
    print('Training sample Macro-F1:',report['macro_f1'], '| cases:',len(selected))
    return report

def export_test(name,threshold=0.5):
    folder=run_folder(name); progress=read_json(folder/'progress.json')
    if not progress['complete']:
        print('This run is paused. Exporting its current best development checkpoint.',flush=True)
    device='cuda' if torch.cuda.is_available() else 'cpu'
    model,spec,saved=load_trained(name,device)
    cfg=spec['config']
    if not 0<threshold<1: raise ValueError('Threshold must lie between zero and one')
    signature=dict(checkpoint=digest_file(folder/'best.pt'),test=object_hash(CASE_INDEX['test']),
                   test_tokens=digest_file(TOKEN_FOLDERS[(spec['model'],'test')]/'tokens.bin'),threshold=threshold)
    out=folder/'test'/object_hash(signature)[:12]; out.mkdir(parents=True,exist_ok=True)
    try:
        logits=predict_logits(model,CaseDataset(spec['model'],'test',cfg),
            cfg['micro_batch_per_gpu']*max(1,torch.cuda.device_count()),device,'Test cases')
    finally:
        del model,saved
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    probabilities=logits.softmax(-1)[:,1].numpy()
    predictions=[dict(id=r['id'],pred=int(p>=threshold)) for r,p in zip(CASE_INDEX['test'],probabilities)]
    validate_predictions(predictions,CASE_INDEX['test'])
    write_json(out/'prediction.json',predictions)
    write_json(out/'probabilities.json',[dict(id=r['id'],probability=float(p)) for r,p in zip(CASE_INDEX['test'],probabilities)])
    write_json(out/'generation.json',signature)
    archive=out/f'{name}_submission.zip'
    with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
        z.write(out/'prediction.json','prediction.json')
    with zipfile.ZipFile(archive) as z:
        assert z.namelist()==['prediction.json']
        validate_predictions(json.loads(z.read('prediction.json')),CASE_INDEX['test'])
    payload=base64.b64encode(archive.read_bytes()).decode()
    display(HTML(f'<a download="{html.escape(archive.name)}" href="data:application/zip;base64,{payload}">Download {html.escape(archive.name)}</a>'))
    print(archive)
    return archive

def completed_runs():
    rows=[]
    for path in (OUTPUT_ROOT/'runs').glob('*/best.pt'):
        folder=path.parent
        if not (folder/'progress.json').exists(): continue
        progress=read_json(folder/'progress.json'); spec=read_json(folder/'config.json')
        rows.append(dict(name=folder.name,model=spec['model'],method=spec['method'],
                    representation=spec['config']['representation'],best_macro_f1=progress['best_macro_f1'],complete=progress['complete']))
    rows.sort(key=lambda r:r['best_macro_f1'],reverse=True)
    return rows

def choose_teacher():
    rows=[r for r in completed_runs() if r['complete'] and r['method'] not in ('kd','control')]
    if not rows: raise ValueError('Complete at least one supervised run before distillation')
    print('Teacher:',rows[0])
    return rows[0]['name']

def cache_teacher(name):
    folder=run_folder(name); progress=read_json(folder/'progress.json')
    if not progress['complete']: raise ValueError('Finish the teacher before caching it')
    spec=read_json(folder/'config.json')
    reference=dict(name=name,checkpoint=digest_file(folder/'best.pt'),cases=object_hash(CASE_INDEX['train']),
                   tokens=digest_file(TOKEN_FOLDERS[(spec['model'],'train')]/'tokens.bin'))
    cache=OUTPUT_ROOT/'teacher_cache'/object_hash(reference)[:16]
    cache.mkdir(parents=True,exist_ok=True)
    if (cache/'logits.pt').exists():
        saved=torch.load(cache/'logits.pt',map_location='cpu',weights_only=False)
        if saved['reference']!=reference or saved['logits'].shape!=(len(CASE_INDEX['train']),2):
            raise ValueError('Teacher cache mismatch')
        return saved['logits'],dict(reference,parameters=saved['parameters'])
    device='cuda' if torch.cuda.is_available() else 'cpu'
    model,spec,saved=load_trained(name,device)
    try:
        cfg=spec['config']; parameters=parameter_report(model)['total']
        logits=predict_logits(model,CaseDataset(spec['model'],'train',cfg),
            cfg['micro_batch_per_gpu']*max(1,torch.cuda.device_count()),device,'Cache teacher on training cases')
        save_torch(cache/'logits.pt',dict(reference=reference,logits=logits,parameters=parameters))
    finally:
        del model,saved
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    return logits,dict(reference,parameters=parameters)


## 11. Ending-only baselines

### Model 1 — full fine-tuning, ending only

Artifacts: `phase4_cjpe/runs/model1_full_tail`.


In [27]:
register('model1_full_tail', 'model1', 'full')
coverage_report('model1', EXPERIMENTS['model1_full_tail']['config'])
capacity_check('model1_full_tail')


{
  "train": {
    "token_coverage": 0.23531127943793123,
    "mean_chunks": 1.0,
    "max_chunks": 1
  },
  "dev": {
    "token_coverage": 0.19823039756116398,
    "mean_chunks": 1.0,
    "max_chunks": 1
  },
  "test": {
    "token_coverage": 0.19616359335810005,
    "mean_chunks": 1.0,
    "max_chunks": 1
  }
}
Saved capacity selection: {'micro_batch_per_gpu': 4, 'activation_checkpointing': False, 'seconds_per_update': 0.3469445059999998, 'memory': [{'gpu': 0, 'peak_reserved_gib': 3.4296875, 'peak_tensor_gib': 3.196878433227539, 'headroom_gib': 10.90704345703125}, {'gpu': 1, 'peak_reserved_gib': 2.408203125, 'peak_tensor_gib': 2.293060779571533, 'headroom_gib': 11.94219970703125}], 'passed': True}


{'micro_batch_per_gpu': 4,
 'activation_checkpointing': False,
 'seconds_per_update': 0.3469445059999998,
 'memory': [{'gpu': 0,
   'peak_reserved_gib': 3.4296875,
   'peak_tensor_gib': 3.196878433227539,
   'headroom_gib': 10.90704345703125},
  {'gpu': 1,
   'peak_reserved_gib': 2.408203125,
   'peak_tensor_gib': 2.293060779571533,
   'headroom_gib': 11.94219970703125}],
 'passed': True}

In [28]:
model1_full_tail_report = train_run('model1_full_tail')
print(model1_full_tail_report)


Parameters: {'total': 120008707, 'trainable': 120008707, 'adapted_modules': 0}
Resumed: {'epoch': 4, 'position': 0, 'step': 15828, 'best_macro_f1': 0.7039768357236573, 'bad_epochs': 3, 'complete': True, 'skipped': 9, 'stop_reason': 'early_stopping'}
{'name': 'model1_full_tail', 'epoch': 4, 'position': 0, 'step': 15828, 'best_macro_f1': 0.7039768357236573, 'bad_epochs': 3, 'complete': True, 'skipped': 9, 'stop_reason': 'early_stopping', 'parameters': {'total': 120008707, 'trainable': 120008707, 'adapted_modules': 0}, 'session_seconds': 1.4569999962077418e-05}
{'name': 'model1_full_tail', 'epoch': 4, 'position': 0, 'step': 15828, 'best_macro_f1': 0.7039768357236573, 'bad_epochs': 3, 'complete': True, 'skipped': 9, 'stop_reason': 'early_stopping', 'parameters': {'total': 120008707, 'trainable': 120008707, 'adapted_modules': 0}, 'session_seconds': 1.4569999962077418e-05}


In [29]:
model1_full_tail_validation = validate_run('model1_full_tail')
model1_full_tail_training_sample = training_sample_report('model1_full_tail')


Evaluate:   0%|          | 0/314 [00:00<?, ?it/s]

Examples: 2511
Macro-F1: 0.703977
Precision: 0.802273
Recall: 0.560763

0 {'precision': 0.6609442060085837, 'recall': 0.8610223642172524, 'f1-score': 0.7478321193201526, 'support': 1252.0}
1 {'precision': 0.8022727272727272, 'recall': 0.5607625099285147, 'f1-score': 0.6601215521271622, 'support': 1259.0}


Training sample:   0%|          | 0/16 [00:00<?, ?it/s]

Training sample Macro-F1: 0.716256157635468 | cases: 128


In [30]:
model1_full_tail_submission = export_test('model1_full_tail')


Test cases:   0%|          | 0/190 [00:00<?, ?it/s]

/kaggle/working/phase4_cjpe/runs/model1_full_tail/test/62a19bde3887/model1_full_tail_submission.zip


### Model 3 — full fine-tuning, ending only

Artifacts: `phase4_cjpe/runs/model3_full_tail`.


In [31]:
register('model3_full_tail', 'model3', 'full')
coverage_report('model3', EXPERIMENTS['model3_full_tail']['config'])
capacity_check('model3_full_tail')


{
  "train": {
    "token_coverage": 0.23531127943793123,
    "mean_chunks": 1.0,
    "max_chunks": 1
  },
  "dev": {
    "token_coverage": 0.19823039756116398,
    "mean_chunks": 1.0,
    "max_chunks": 1
  },
  "test": {
    "token_coverage": 0.19616359335810005,
    "mean_chunks": 1.0,
    "max_chunks": 1
  }
}
Testing 1 cases/GPU; checkpointing=True
{'micro_batch_per_gpu': 1, 'activation_checkpointing': True, 'seconds_per_update': 1.0199643950001018, 'memory': [{'gpu': 0, 'peak_reserved_gib': 3.12890625, 'peak_tensor_gib': 2.693631172180176, 'headroom_gib': 11.20782470703125}, {'gpu': 1, 'peak_reserved_gib': 0.65234375, 'peak_tensor_gib': 0.5456690788269043, 'headroom_gib': 13.69805908203125}], 'passed': True}
Testing 2 cases/GPU; checkpointing=True
{'micro_batch_per_gpu': 2, 'activation_checkpointing': True, 'seconds_per_update': 0.5791795080000384, 'memory': [{'gpu': 0, 'peak_reserved_gib': 3.177734375, 'peak_tensor_gib': 2.7051477432250977, 'headroom_gib': 11.15899658203125}, {'g

{'micro_batch_per_gpu': 4,
 'activation_checkpointing': False,
 'seconds_per_update': 0.36589301300000443,
 'memory': [{'gpu': 0,
   'peak_reserved_gib': 3.822265625,
   'peak_tensor_gib': 3.4986610412597656,
   'headroom_gib': 10.51446533203125},
  {'gpu': 1,
   'peak_reserved_gib': 2.701171875,
   'peak_tensor_gib': 2.5750346183776855,
   'headroom_gib': 11.64923095703125}],
 'passed': True}

In [32]:
model3_full_tail_report = train_run('model3_full_tail')
print(model3_full_tail_report)


Parameters: {'total': 118566915, 'trainable': 118566915, 'adapted_modules': 0}


Evaluate:   0%|          | 0/314 [00:00<?, ?it/s]

{'step': 0, 'epoch': 0, 'validation_macro_f1': 0.40731764709300045}
{'step': 1, 'epoch': 0, 'loss': 0.5062001943588257, 'lr': 2.1063717746182202e-08, 'gradient_norm': 9.461421966552734, 'skipped': 0, 'elapsed_minutes': 0.6483352402499993}
{'step': 20, 'epoch': 0, 'loss': 0.8169525861740112, 'lr': 2.2116903633491313e-07, 'gradient_norm': 13.203652381896973, 'skipped': 0, 'elapsed_minutes': 0.7703798465999986}
{'step': 40, 'epoch': 0, 'loss': 0.8042721152305603, 'lr': 4.3180621379673515e-07, 'gradient_norm': 11.537520408630371, 'skipped': 0, 'elapsed_minutes': 0.8999180883166654}
{'step': 60, 'epoch': 0, 'loss': 0.8954111337661743, 'lr': 6.424433912585572e-07, 'gradient_norm': 15.530659675598145, 'skipped': 0, 'elapsed_minutes': 1.0294650164166663}
AMP update skipped; retrying the same cases
{'step': 80, 'epoch': 0, 'loss': 0.806858479976654, 'lr': 8.530805687203792e-07, 'gradient_norm': 14.762018203735352, 'skipped': 1, 'elapsed_minutes': 1.1650566116499987}
{'step': 100, 'epoch': 0, 'l

Evaluate:   0%|          | 0/314 [00:00<?, ?it/s]

{'step': 3957, 'epoch': 1, 'validation_macro_f1': 0.7073816872427984}
{'step': 3960, 'epoch': 1, 'loss': 0.23636749386787415, 'lr': 1.9787785140473197e-05, 'gradient_norm': 1.994919776916504, 'skipped': 3, 'elapsed_minutes': 27.800602243750003}
{'step': 3980, 'epoch': 1, 'loss': 0.8505431413650513, 'lr': 1.978366313902352e-05, 'gradient_norm': 8.98142147064209, 'skipped': 3, 'elapsed_minutes': 27.929293511916672}
{'step': 4000, 'epoch': 1, 'loss': 0.7341994047164917, 'lr': 1.9779501976286738e-05, 'gradient_norm': 4.707005500793457, 'skipped': 3, 'elapsed_minutes': 28.057877981733334}
{'step': 4020, 'epoch': 1, 'loss': 0.35976171493530273, 'lr': 1.9775301670815067e-05, 'gradient_norm': 2.7421936988830566, 'skipped': 3, 'elapsed_minutes': 28.213835172650004}
{'step': 4040, 'epoch': 1, 'loss': 0.31254613399505615, 'lr': 1.9771062241335256e-05, 'gradient_norm': 2.786405563354492, 'skipped': 3, 'elapsed_minutes': 28.342488450883334}
{'step': 4060, 'epoch': 1, 'loss': 0.5632402896881104, 'lr

Evaluate:   0%|          | 0/314 [00:00<?, ?it/s]

... (3 outputs left out)

Evaluate:   0%|          | 0/314 [00:00<?, ?it/s]

{'step': 15828, 'epoch': 4, 'validation_macro_f1': 0.6134186737214475}
{'step': 15840, 'epoch': 4, 'loss': 0.006847257725894451, 'lr': 1.1889334620032557e-05, 'gradient_norm': 0.3768326938152313, 'skipped': 8, 'elapsed_minutes': 109.0201173283}
{'step': 15860, 'epoch': 4, 'loss': 0.5509638786315918, 'lr': 1.1870422173792087e-05, 'gradient_norm': 9.872862815856934, 'skipped': 8, 'elapsed_minutes': 109.14738537316666}
{'step': 15880, 'epoch': 4, 'loss': 0.006488940212875605, 'lr': 1.1851505846841242e-05, 'gradient_norm': 1.0286815166473389, 'skipped': 8, 'elapsed_minutes': 109.27457022596667}
{'step': 15900, 'epoch': 4, 'loss': 0.0036985035985708237, 'lr': 1.1832585723517015e-05, 'gradient_norm': 0.12730754911899567, 'skipped': 8, 'elapsed_minutes': 109.40184471181666}
{'step': 15920, 'epoch': 4, 'loss': 0.0022923096548765898, 'lr': 1.1813661888173327e-05, 'gradient_norm': 0.0401599295437336, 'skipped': 8, 'elapsed_minutes': 109.55417145051666}
{'step': 15940, 'epoch': 4, 'loss': 0.00288

Evaluate:   0%|          | 0/314 [00:00<?, ?it/s]

{'step': 19785, 'epoch': 5, 'validation_macro_f1': 0.6593453573728598}
{'name': 'model3_full_tail', 'epoch': 5, 'position': 0, 'step': 19785, 'best_macro_f1': 0.7091251771484329, 'bad_epochs': 3, 'complete': True, 'skipped': 10, 'stop_reason': 'early_stopping', 'parameters': {'total': 118566915, 'trainable': 118566915, 'adapted_modules': 0}, 'session_seconds': 8140.671781861}
{'name': 'model3_full_tail', 'epoch': 5, 'position': 0, 'step': 19785, 'best_macro_f1': 0.7091251771484329, 'bad_epochs': 3, 'complete': True, 'skipped': 10, 'stop_reason': 'early_stopping', 'parameters': {'total': 118566915, 'trainable': 118566915, 'adapted_modules': 0}, 'session_seconds': 8140.671781861}


In [33]:
model3_full_tail_validation = validate_run('model3_full_tail')
model3_full_tail_training_sample = training_sample_report('model3_full_tail')


Evaluate:   0%|          | 0/314 [00:00<?, ?it/s]

Examples: 2511
Macro-F1: 0.709125
Precision: 0.792432
Recall: 0.582208

0 {'precision': 0.6683480453972257, 'recall': 0.8466453674121406, 'f1-score': 0.7470049330514447, 'support': 1252.0}
1 {'precision': 0.7924324324324324, 'recall': 0.5822081016679904, 'f1-score': 0.6712454212454212, 'support': 1259.0}


Training sample:   0%|          | 0/16 [00:00<?, ?it/s]

Training sample Macro-F1: 0.8745098039215686 | cases: 128


In [34]:
model3_full_tail_submission = export_test('model3_full_tail')


Test cases:   0%|          | 0/190 [00:00<?, ?it/s]

/kaggle/working/phase4_cjpe/runs/model3_full_tail/test/953b04d53ced/model3_full_tail_submission.zip


## 12. Compare document representations

The next cell chooses the stronger completed ending-only baseline. It then registers two additional runs on that same backbone. Each representation starts again from Phase 1. Keep the same seed and adaptation settings for this comparison. If only one baseline is complete, it is used provisionally.


In [35]:
candidates=[r for r in completed_runs() if r['name'] in ('model1_full_tail','model3_full_tail') and r['complete']]
if not candidates: raise ValueError('Complete an ending-only baseline first')
EXPLORE_MODEL=candidates[0]['model']
print('Representation comparison backbone:', EXPLORE_MODEL)
register('representation_head_tail',EXPLORE_MODEL,'full',representation='head_tail')
register('representation_hierarchical',EXPLORE_MODEL,'full',representation='hierarchical',aggregator='bigru')


Representation comparison backbone: model3


'representation_hierarchical'

### Beginning and ending


In [36]:
coverage_report(EXPLORE_MODEL,EXPERIMENTS['representation_head_tail']['config'])
capacity_check('representation_head_tail')


{
  "train": {
    "token_coverage": 0.4130370247492767,
    "mean_chunks": 1.7690898177108014,
    "max_chunks": 2
  },
  "dev": {
    "token_coverage": 0.37983429178924116,
    "mean_chunks": 1.9609717244125846,
    "max_chunks": 2
  },
  "test": {
    "token_coverage": 0.37601313961338445,
    "mean_chunks": 1.96044825313118,
    "max_chunks": 2
  }
}
Testing 1 cases/GPU; checkpointing=True
{'micro_batch_per_gpu': 1, 'activation_checkpointing': True, 'seconds_per_update': 1.152191192999453, 'memory': [{'gpu': 0, 'peak_reserved_gib': 3.177734375, 'peak_tensor_gib': 2.703634262084961, 'headroom_gib': 11.15899658203125}, {'gpu': 1, 'peak_reserved_gib': 0.734375, 'peak_tensor_gib': 0.6079421043395996, 'headroom_gib': 13.61602783203125}], 'passed': True}
Testing 2 cases/GPU; checkpointing=True
{'micro_batch_per_gpu': 2, 'activation_checkpointing': True, 'seconds_per_update': 0.8740353230004985, 'memory': [{'gpu': 0, 'peak_reserved_gib': 3.146484375, 'peak_tensor_gib': 2.703761100769043, 

{'micro_batch_per_gpu': 4,
 'activation_checkpointing': False,
 'seconds_per_update': 0.628355372000442,
 'memory': [{'gpu': 0,
   'peak_reserved_gib': 5.62890625,
   'peak_tensor_gib': 5.367249488830566,
   'headroom_gib': 8.70782470703125},
  {'gpu': 1,
   'peak_reserved_gib': 4.59765625,
   'peak_tensor_gib': 4.47730016708374,
   'headroom_gib': 9.75274658203125}],
 'passed': True}

In [37]:
representation_head_tail_report=train_run('representation_head_tail')
print(representation_head_tail_report)


Parameters: {'total': 118566915, 'trainable': 118566915, 'adapted_modules': 0}


Evaluate:   0%|          | 0/314 [00:00<?, ?it/s]

{'step': 0, 'epoch': 0, 'validation_macro_f1': 0.4199371769047653}
{'step': 1, 'epoch': 0, 'loss': 0.49262547492980957, 'lr': 2.1063717746182202e-08, 'gradient_norm': 10.776018142700195, 'skipped': 0, 'elapsed_minutes': 1.1036320789333331}
{'step': 20, 'epoch': 0, 'loss': 0.7278590202331543, 'lr': 2.2116903633491313e-07, 'gradient_norm': 13.051340103149414, 'skipped': 0, 'elapsed_minutes': 1.286572157566661}
{'step': 40, 'epoch': 0, 'loss': 0.6923487782478333, 'lr': 4.3180621379673515e-07, 'gradient_norm': 12.399443626403809, 'skipped': 0, 'elapsed_minutes': 1.479818835666659}
{'step': 60, 'epoch': 0, 'loss': 0.8555001616477966, 'lr': 6.424433912585572e-07, 'gradient_norm': 17.014873504638672, 'skipped': 0, 'elapsed_minutes': 1.6708893588500056}
{'step': 80, 'epoch': 0, 'loss': 0.7612327337265015, 'lr': 8.530805687203792e-07, 'gradient_norm': 16.820592880249023, 'skipped': 0, 'elapsed_minutes': 1.8673171985500023}
{'step': 100, 'epoch': 0, 'loss': 0.5209028720855713, 'lr': 1.0637177461

Evaluate:   0%|          | 0/314 [00:00<?, ?it/s]

{'step': 3957, 'epoch': 1, 'validation_macro_f1': 0.6792274088068451}
{'step': 3960, 'epoch': 1, 'loss': 0.3941039741039276, 'lr': 1.9787785140473197e-05, 'gradient_norm': 3.283726215362549, 'skipped': 3, 'elapsed_minutes': 41.84345738163335}
{'step': 3980, 'epoch': 1, 'loss': 0.8064774870872498, 'lr': 1.978366313902352e-05, 'gradient_norm': 10.02287483215332, 'skipped': 3, 'elapsed_minutes': 42.03729927289999}
{'step': 4000, 'epoch': 1, 'loss': 0.6957566142082214, 'lr': 1.9779501976286738e-05, 'gradient_norm': 6.330764293670654, 'skipped': 3, 'elapsed_minutes': 42.235007769000006}
{'step': 4020, 'epoch': 1, 'loss': 0.3563989996910095, 'lr': 1.9775301670815067e-05, 'gradient_norm': 3.1853556632995605, 'skipped': 3, 'elapsed_minutes': 42.45953774556668}
{'step': 4040, 'epoch': 1, 'loss': 0.3905298113822937, 'lr': 1.9771062241335256e-05, 'gradient_norm': 2.542799711227417, 'skipped': 3, 'elapsed_minutes': 42.649903275}
{'step': 4060, 'epoch': 1, 'loss': 0.4778924584388733, 'lr': 1.976678

Evaluate:   0%|          | 0/314 [00:00<?, ?it/s]

{'step': 7914, 'epoch': 2, 'validation_macro_f1': 0.6695226221796757}
{'step': 7920, 'epoch': 2, 'loss': 0.275550901889801, 'lr': 1.8242085245621067e-05, 'gradient_norm': 2.718909978866577, 'skipped': 5, 'elapsed_minutes': 82.38115491231669}
{'step': 7940, 'epoch': 2, 'loss': 0.8521363139152527, 'lr': 1.8230786475304224e-05, 'gradient_norm': 13.527647972106934, 'skipped': 5, 'elapsed_minutes': 82.57414121219999}
{'step': 7960, 'epoch': 2, 'loss': 0.24306271970272064, 'lr': 1.8219455467081544e-05, 'gradient_norm': 7.349783420562744, 'skipped': 5, 'elapsed_minutes': 82.77099673793333}
{'step': 7980, 'epoch': 2, 'loss': 0.27438461780548096, 'lr': 1.8208092271471462e-05, 'gradient_norm': 9.153995513916016, 'skipped': 5, 'elapsed_minutes': 82.96755897241668}
{'step': 8000, 'epoch': 2, 'loss': 0.34004873037338257, 'lr': 1.8196696939135902e-05, 'gradient_norm': 5.050528049468994, 'skipped': 5, 'elapsed_minutes': 83.15927716676667}
{'step': 8020, 'epoch': 2, 'loss': 0.3797898292541504, 'lr': 1

Evaluate:   0%|          | 0/314 [00:00<?, ?it/s]

{'step': 11871, 'epoch': 3, 'validation_macro_f1': 0.6328773428457326}
{'step': 11880, 'epoch': 3, 'loss': 0.06096208840608597, 'lr': 1.5448885712281884e-05, 'gradient_norm': 1.6084041595458984, 'skipped': 6, 'elapsed_minutes': 122.79859553543335}
{'step': 11900, 'epoch': 3, 'loss': 0.08768919110298157, 'lr': 1.5432356465002424e-05, 'gradient_norm': 7.076566696166992, 'skipped': 6, 'elapsed_minutes': 122.99585047951669}
{'step': 11920, 'epoch': 3, 'loss': 0.09859801828861237, 'lr': 1.5415807456402282e-05, 'gradient_norm': 68.28760528564453, 'skipped': 6, 'elapsed_minutes': 123.21614346948333}
{'step': 11940, 'epoch': 3, 'loss': 0.011879688128829002, 'lr': 1.5399238760263936e-05, 'gradient_norm': 0.2671550512313843, 'skipped': 6, 'elapsed_minutes': 123.40397240211665}
{'step': 11960, 'epoch': 3, 'loss': 0.27066874504089355, 'lr': 1.5382650450457645e-05, 'gradient_norm': 6.852922439575195, 'skipped': 6, 'elapsed_minutes': 123.59568786276668}
{'step': 11980, 'epoch': 3, 'loss': 0.51128888

Evaluate:   0%|          | 0/314 [00:00<?, ?it/s]

{'step': 15828, 'epoch': 4, 'validation_macro_f1': 0.6384583158776707}
{'name': 'representation_head_tail', 'epoch': 4, 'position': 0, 'step': 15828, 'best_macro_f1': 0.6792274088068451, 'bad_epochs': 3, 'complete': True, 'skipped': 7, 'stop_reason': 'early_stopping', 'parameters': {'total': 118566915, 'trainable': 118566915, 'adapted_modules': 0}, 'session_seconds': 9762.207116593001}
{'name': 'representation_head_tail', 'epoch': 4, 'position': 0, 'step': 15828, 'best_macro_f1': 0.6792274088068451, 'bad_epochs': 3, 'complete': True, 'skipped': 7, 'stop_reason': 'early_stopping', 'parameters': {'total': 118566915, 'trainable': 118566915, 'adapted_modules': 0}, 'session_seconds': 9762.207116593001}


In [38]:
representation_head_tail_validation=validate_run('representation_head_tail')
representation_head_tail_training_sample=training_sample_report('representation_head_tail')


Evaluate:   0%|          | 0/314 [00:00<?, ?it/s]

Examples: 2511
Macro-F1: 0.679227
Precision: 0.808974
Recall: 0.501191

0 {'precision': 0.6372039283651069, 'recall': 0.8809904153354633, 'f1-score': 0.7395239691585652, 'support': 1252.0}
1 {'precision': 0.808974358974359, 'recall': 0.5011914217633042, 'f1-score': 0.618930848455125, 'support': 1259.0}


Training sample:   0%|          | 0/16 [00:00<?, ?it/s]

Training sample Macro-F1: 0.7733129770992366 | cases: 128


In [39]:
representation_head_tail_submission=export_test('representation_head_tail')


Test cases:   0%|          | 0/190 [00:00<?, ?it/s]

/kaggle/working/phase4_cjpe/runs/representation_head_tail/test/a70b3456b894/representation_head_tail_submission.zip


### Ordered chunks with BiGRU aggregation


In [40]:
coverage_report(EXPLORE_MODEL,EXPERIMENTS['representation_hierarchical']['config'])
capacity_check('representation_hierarchical')


{
  "train": {
    "token_coverage": 0.7310587677132954,
    "mean_chunks": 3.3858402047199316,
    "max_chunks": 6
  },
  "dev": {
    "token_coverage": 0.7530187319642787,
    "mean_chunks": 4.410195141377937,
    "max_chunks": 6
  },
  "test": {
    "token_coverage": 0.7498152225866994,
    "mean_chunks": 4.4410019775873435,
    "max_chunks": 6
  }
}
Testing 1 cases/GPU; checkpointing=True
{'micro_batch_per_gpu': 1, 'activation_checkpointing': True, 'seconds_per_update': 2.210217580999597, 'memory': [{'gpu': 0, 'peak_reserved_gib': 3.037109375, 'peak_tensor_gib': 2.700723648071289, 'headroom_gib': 11.29571533203125}, {'gpu': 1, 'peak_reserved_gib': 1.076171875, 'peak_tensor_gib': 0.8820986747741699, 'headroom_gib': 13.27032470703125}], 'passed': True}
Testing 2 cases/GPU; checkpointing=True
{'micro_batch_per_gpu': 2, 'activation_checkpointing': True, 'seconds_per_update': 2.0586612689985486, 'memory': [{'gpu': 0, 'peak_reserved_gib': 3.388671875, 'peak_tensor_gib': 2.778856754302978

{'micro_batch_per_gpu': 2,
 'activation_checkpointing': False,
 'seconds_per_update': 1.7056882889992266,
 'memory': [{'gpu': 0,
   'peak_reserved_gib': 8.134765625,
   'peak_tensor_gib': 7.709447860717773,
   'headroom_gib': 6.19805908203125},
  {'gpu': 1,
   'peak_reserved_gib': 6.484375,
   'peak_tensor_gib': 6.351572513580322,
   'headroom_gib': 7.86212158203125}],
 'passed': True}

In [41]:
representation_hierarchical_report=train_run('representation_hierarchical')
print(representation_hierarchical_report)


Parameters: {'total': 118641411, 'trainable': 118641411, 'adapted_modules': 0}


Evaluate:   0%|          | 0/628 [00:00<?, ?it/s]

{'step': 0, 'epoch': 0, 'validation_macro_f1': 0.5151478562931592}
{'step': 1, 'epoch': 0, 'loss': 1.2209776043891907, 'lr': 2.1063717746182202e-08, 'gradient_norm': 25.191560745239258, 'skipped': 0, 'elapsed_minutes': 2.5834543626500213}
{'step': 20, 'epoch': 0, 'loss': 0.8422944396734238, 'lr': 2.2116903633491313e-07, 'gradient_norm': 18.2137393951416, 'skipped': 0, 'elapsed_minutes': 2.9522217704833262}
{'step': 40, 'epoch': 0, 'loss': 0.8005003929138184, 'lr': 4.3180621379673515e-07, 'gradient_norm': 16.98900604248047, 'skipped': 0, 'elapsed_minutes': 3.359918694316669}
{'step': 60, 'epoch': 0, 'loss': 0.668210506439209, 'lr': 6.424433912585572e-07, 'gradient_norm': 14.508295059204102, 'skipped': 0, 'elapsed_minutes': 3.763209652016667}
{'step': 80, 'epoch': 0, 'loss': 0.8133760988712311, 'lr': 8.530805687203792e-07, 'gradient_norm': 19.853342056274414, 'skipped': 0, 'elapsed_minutes': 4.156304501549979}
{'step': 100, 'epoch': 0, 'loss': 0.8427651822566986, 'lr': 1.0637177461822013

Evaluate:   0%|          | 0/628 [00:00<?, ?it/s]

{'step': 3957, 'epoch': 1, 'validation_macro_f1': 0.7052111415890656}
{'step': 3960, 'epoch': 1, 'loss': 0.33959659934043884, 'lr': 1.9787785140473197e-05, 'gradient_norm': 3.3247358798980713, 'skipped': 1, 'elapsed_minutes': 86.03025459758331}
{'step': 3980, 'epoch': 1, 'loss': 0.7306368350982666, 'lr': 1.978366313902352e-05, 'gradient_norm': 5.096996784210205, 'skipped': 1, 'elapsed_minutes': 86.40692734533332}
{'step': 4000, 'epoch': 1, 'loss': 0.5778795927762985, 'lr': 1.9779501976286738e-05, 'gradient_norm': 6.230467319488525, 'skipped': 1, 'elapsed_minutes': 86.8194731447833}
{'step': 4020, 'epoch': 1, 'loss': 0.4528094679117203, 'lr': 1.9775301670815067e-05, 'gradient_norm': 3.8393287658691406, 'skipped': 1, 'elapsed_minutes': 87.25051988878332}
{'step': 4040, 'epoch': 1, 'loss': 0.366404689848423, 'lr': 1.9771062241335256e-05, 'gradient_norm': 2.3861470222473145, 'skipped': 1, 'elapsed_minutes': 87.6351858389833}
{'step': 4060, 'epoch': 1, 'loss': 0.4117431864142418, 'lr': 1.97

Evaluate:   0%|          | 0/628 [00:00<?, ?it/s]

{'step': 7914, 'epoch': 2, 'validation_macro_f1': 0.6971937398812735}
{'step': 7920, 'epoch': 2, 'loss': 0.318154938519001, 'lr': 1.8242085245621067e-05, 'gradient_norm': 2.031096935272217, 'skipped': 2, 'elapsed_minutes': 169.00595025396663}
{'step': 7940, 'epoch': 2, 'loss': 0.8240925669670105, 'lr': 1.8230786475304224e-05, 'gradient_norm': 4.618477821350098, 'skipped': 2, 'elapsed_minutes': 169.39244062878333}
{'step': 7960, 'epoch': 2, 'loss': 0.36964452266693115, 'lr': 1.8219455467081544e-05, 'gradient_norm': 6.777817249298096, 'skipped': 2, 'elapsed_minutes': 169.80842112906666}
{'step': 7980, 'epoch': 2, 'loss': 0.9575856178998947, 'lr': 1.8208092271471462e-05, 'gradient_norm': 17.243595123291016, 'skipped': 2, 'elapsed_minutes': 170.22051025078332}
{'step': 8000, 'epoch': 2, 'loss': 0.20153386890888214, 'lr': 1.8196696939135902e-05, 'gradient_norm': 4.202632904052734, 'skipped': 2, 'elapsed_minutes': 170.65167181795}
{'step': 8020, 'epoch': 2, 'loss': 0.7350002229213715, 'lr': 

Evaluate:   0%|          | 0/628 [00:00<?, ?it/s]

{'step': 11871, 'epoch': 3, 'validation_macro_f1': 0.6622116908845672}
{'step': 11880, 'epoch': 3, 'loss': 0.19379755854606628, 'lr': 1.5448885712281884e-05, 'gradient_norm': 3.4898247718811035, 'skipped': 4, 'elapsed_minutes': 251.70618137700004}
{'step': 11900, 'epoch': 3, 'loss': 0.04119025357067585, 'lr': 1.5432356465002424e-05, 'gradient_norm': 0.8865031599998474, 'skipped': 4, 'elapsed_minutes': 252.11085388581668}
{'step': 11920, 'epoch': 3, 'loss': 0.03307057544589043, 'lr': 1.5415807456402282e-05, 'gradient_norm': 1.0002485513687134, 'skipped': 4, 'elapsed_minutes': 252.55426210566668}
{'step': 11940, 'epoch': 3, 'loss': 0.0721132941544056, 'lr': 1.5399238760263936e-05, 'gradient_norm': 2.766295909881592, 'skipped': 4, 'elapsed_minutes': 252.9448104783166}
{'step': 11960, 'epoch': 3, 'loss': 0.3309032302349806, 'lr': 1.5382650450457645e-05, 'gradient_norm': 12.404130935668945, 'skipped': 4, 'elapsed_minutes': 253.34726992795004}
{'step': 11980, 'epoch': 3, 'loss': 0.0372880380

Evaluate:   0%|          | 0/628 [00:00<?, ?it/s]

{'step': 15828, 'epoch': 4, 'validation_macro_f1': 0.6454434500637631}
{'name': 'representation_hierarchical', 'epoch': 4, 'position': 0, 'step': 15828, 'best_macro_f1': 0.7052111415890656, 'bad_epochs': 3, 'complete': True, 'skipped': 6, 'stop_reason': 'early_stopping', 'parameters': {'total': 118641411, 'trainable': 118641411, 'adapted_modules': 0}, 'session_seconds': 20021.420376707}
{'name': 'representation_hierarchical', 'epoch': 4, 'position': 0, 'step': 15828, 'best_macro_f1': 0.7052111415890656, 'bad_epochs': 3, 'complete': True, 'skipped': 6, 'stop_reason': 'early_stopping', 'parameters': {'total': 118641411, 'trainable': 118641411, 'adapted_modules': 0}, 'session_seconds': 20021.420376707}


In [42]:
representation_hierarchical_validation=validate_run('representation_hierarchical')
representation_hierarchical_training_sample=training_sample_report('representation_hierarchical')


Evaluate:   0%|          | 0/628 [00:00<?, ?it/s]

Examples: 2511
Macro-F1: 0.705211
Precision: 0.831089
Recall: 0.539317

0 {'precision': 0.6576151121605667, 'recall': 0.889776357827476, 'f1-score': 0.7562797012898846, 'support': 1252.0}
1 {'precision': 0.8310893512851897, 'recall': 0.5393169181890389, 'f1-score': 0.6541425818882466, 'support': 1259.0}


Training sample:   0%|          | 0/32 [00:00<?, ?it/s]

Training sample Macro-F1: 0.7576793893129772 | cases: 128


In [43]:
representation_hierarchical_submission=export_test('representation_hierarchical')


Test cases:   0%|          | 0/380 [00:00<?, ?it/s]

/kaggle/working/phase4_cjpe/runs/representation_hierarchical/test/9a7508d0dc0d/representation_hierarchical_submission.zip


## 13. Required adaptation experiments

This selection uses the best completed representation on the comparison backbone. The same document policy is carried to the remaining methods. Scores on another backbone can differ; these settings are a shared starting point. Save this choice before launching separate Kaggle sessions. To reproduce it elsewhere, set `DOCUMENT_POLICY` explicitly to the printed dictionary. The actual values are stored in every run configuration.


In [44]:
rows=[r for r in completed_runs() if r['complete'] and r['model']==EXPLORE_MODEL and r['method']=='full']
if not rows: raise ValueError('No completed representation experiment')
selected=read_json(run_folder(rows[0]['name'])/'config.json')['config']
DOCUMENT_POLICY={k:selected[k] for k in ('representation','aggregator','chunk_tokens','overlap','max_chunks')}
print('Document policy:',DOCUMENT_POLICY)
write_json(ASSET_DIR/'selected_document_policy.json',DOCUMENT_POLICY)


Document policy: {'representation': 'tail', 'aggregator': 'attention', 'chunk_tokens': 1024, 'overlap': 128, 'max_chunks': 6}


### Model 1 · LoRMA

This run uses the final 1,024 tokens with attention pooling. It saves to `phase4_cjpe/runs/model1_lorma/`.

In [26]:
import gc
import math
import time
import numpy as np
import torch
import torch.nn.functional as F


def capacity_check(name):
    spec = EXPERIMENTS[name]
    cfg = spec["config"]
    folder = run_folder(name)

    if not torch.cuda.is_available():
        raise RuntimeError("Enable the Kaggle GPU first.")

    path = folder / "hardware.json"
    signature = hardware_signature(spec)

    if path.exists():
        saved = read_json(path)
        if saved["signature"] != signature:
            raise ValueError(
                "Saved capacity settings differ from this run. "
                "Keep the original configuration when resuming."
            )
        print("Using saved capacity selection:", saved["selected"], flush=True)
        return saved["selected"]

    results = []
    n_gpu = torch.cuda.device_count()

    for checkpointing in (True, False):
        for micro in (1, 2, 4):
            batch_size = micro * n_gpu

            if (
                batch_size > cfg["effective_batch"]
                or cfg["effective_batch"] % batch_size
            ):
                continue

            trial = dict(
                cfg,
                micro_batch_per_gpu=micro,
                activation_checkpointing=checkpointing,
            )

            model = net = optimizer = scaler = None
            x = y = loss = norm = None

            print(
                f"Testing {micro} cases/GPU; "
                f"checkpointing={checkpointing}",
                flush=True,
            )

            try:
                gc.collect()
                torch.cuda.empty_cache()
                seed_everything(cfg["seed"])

                model = make_model(
                    spec["model"],
                    spec["method"],
                    trial,
                    total_updates(cfg),
                    "cuda",
                )
                model.train()
                net = parallel_model(model)
                optimizer = make_optimizer(model, trial)
                scaler = torch.amp.GradScaler(
                    "cuda",
                    init_scale=1024.0,
                )

                chunks = max_case_chunks(cfg)
                length = cfg["chunk_tokens"]

                x = {
                    "ids": torch.randint(
                        4,
                        BASES[spec["model"]]["config"]["vocab_size"],
                        (batch_size, chunks, length),
                        device="cuda",
                    ),
                    "mask": torch.ones(
                        batch_size,
                        chunks,
                        length,
                        dtype=torch.bool,
                        device="cuda",
                    ),
                    "positions": torch.linspace(
                        0, 1, chunks, device="cuda"
                    )[None, :].expand(batch_size, -1),
                }
                y = torch.arange(batch_size, device="cuda") % 2

                for gpu in range(n_gpu):
                    torch.cuda.reset_peak_memory_stats(gpu)

                accumulation = cfg["effective_batch"] // batch_size
                durations = []
                successful = 0
                retries = 0

                while successful < 4:
                    optimizer.zero_grad(set_to_none=True)
                    sync_cuda()
                    started = time.perf_counter()

                    for _ in range(accumulation):
                        with autocast_context("cuda"):
                            loss = (
                                F.cross_entropy(net(**x), y)
                                / accumulation
                            )

                        if not torch.isfinite(loss).item():
                            raise FloatingPointError(
                                "Non-finite forward loss."
                            )

                        scaler.scale(loss).backward()

                    if spec["method"] == "adalora":
                        scaler.scale(
                            adalora_penalty(model)
                        ).backward()

                    scaler.unscale_(optimizer)
                    norm = torch.nn.utils.clip_grad_norm_(
                        model.parameters(),
                        cfg["clip"],
                    )

                    if not torch.isfinite(norm).item():
                        retries += 1
                        old_scale = scaler.get_scale()

                        # Discard this attempt and retry at a lower scale.
                        scaler.update(new_scale=old_scale / 2)
                        optimizer.zero_grad(set_to_none=True)

                        print(
                            f"AMP retry {retries}/12: "
                            f"scale {old_scale:g} -> {old_scale / 2:g}",
                            flush=True,
                        )

                        if retries >= 12:
                            raise FloatingPointError(
                                "Gradients remained non-finite "
                                "after 12 retries."
                            )
                        continue

                    scaler.step(optimizer)
                    scaler.update()

                    if spec["method"] == "adalora":
                        model.encoder.update_and_allocate(
                            successful + 1
                        )

                    sync_cuda()
                    durations.append(time.perf_counter() - started)
                    successful += 1

                memory = []
                for gpu in range(n_gpu):
                    free, total = torch.cuda.mem_get_info(gpu)
                    reserved = torch.cuda.max_memory_reserved(gpu)

                    memory.append({
                        "gpu": gpu,
                        "peak_reserved_gib": reserved / 2**30,
                        "peak_tensor_gib": (
                            torch.cuda.max_memory_allocated(gpu)
                            / 2**30
                        ),
                        "headroom_gib": min(
                            free, total - reserved
                        ) / 2**30,
                    })

                row = {
                    "micro_batch_per_gpu": micro,
                    "activation_checkpointing": checkpointing,
                    "seconds_per_update": float(
                        np.median(durations[1:])
                    ),
                    "amp_retries": retries,
                    "memory": memory,
                    "passed": all(
                        m["headroom_gib"]
                        >= cfg["capacity_headroom_gib"]
                        for m in memory
                    ),
                }

            except (torch.cuda.OutOfMemoryError, FloatingPointError) as error:
                row = {
                    "micro_batch_per_gpu": micro,
                    "activation_checkpointing": checkpointing,
                    "passed": False,
                    "error": str(error),
                }

            finally:
                model = net = optimizer = scaler = None
                x = y = loss = norm = None
                gc.collect()
                torch.cuda.empty_cache()

            results.append(row)
            print(row, flush=True)

    passing = [row for row in results if row["passed"]]
    if not passing:
        raise RuntimeError(
            "No capacity candidate passed. "
            "Check the printed failures before changing the run."
        )

    selected = min(
        passing,
        key=lambda row: row["seconds_per_update"],
    )

    folder.mkdir(parents=True, exist_ok=True)
    write_json(path, {
        "signature": signature,
        "results": results,
        "selected": selected,
    })

    print("Selected:", selected, flush=True)
    return selected


capacity_check.amp_retry_fix = True

required = [
    "EXPERIMENTS", "register", "DEFAULT", "BASES",
    "CASE_INDEX", "TOKEN_FOLDERS", "make_model", "train_run",
]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        "Run the notebook setup/helper cells first. Missing: "
        + ", ".join(missing)
    )

LORMA_RUN = "model1_lorma"

if LORMA_RUN not in EXPERIMENTS:
    register(
        LORMA_RUN,
        "model1",
        "lorma",
        representation="tail",
        chunk_tokens=1024,
        overlap=128,
        max_chunks=6,
        aggregator="attention",
    )

assert capacity_check.amp_retry_fix
print("Corrected capacity check loaded.", flush=True)
print("Run folder:", run_folder(LORMA_RUN), flush=True)

capacity_check(LORMA_RUN)

lorma_report = train_run(LORMA_RUN)
print(lorma_report, flush=True)

Corrected capacity check loaded.
Run folder: /kaggle/working/phase4_cjpe/runs/model1_lorma
Testing 1 cases/GPU; checkpointing=True
{'micro_batch_per_gpu': 1, 'activation_checkpointing': True, 'seconds_per_update': 1.0887117620000026, 'amp_retries': 0, 'memory': [{'gpu': 0, 'peak_reserved_gib': 2.10546875, 'peak_tensor_gib': 1.784705638885498, 'headroom_gib': 12.23126220703125}, {'gpu': 1, 'peak_reserved_gib': 0.599609375, 'peak_tensor_gib': 0.562281608581543, 'headroom_gib': 13.75079345703125}], 'passed': True}
Testing 2 cases/GPU; checkpointing=True
{'micro_batch_per_gpu': 2, 'activation_checkpointing': True, 'seconds_per_update': 0.5946246360003897, 'amp_retries': 0, 'memory': [{'gpu': 0, 'peak_reserved_gib': 2.111328125, 'peak_tensor_gib': 1.7844157218933105, 'headroom_gib': 12.22540283203125}, {'gpu': 1, 'peak_reserved_gib': 0.6796875, 'peak_tensor_gib': 0.6323671340942383, 'headroom_gib': 13.67071533203125}], 'passed': True}
Testing 4 cases/GPU; checkpointing=True
{'micro_batch_pe

Evaluate:   0%|          | 0/314 [00:00<?, ?it/s]

{'step': 0, 'epoch': 0, 'validation_macro_f1': 0.377390021728147}
AMP update skipped; retrying the same cases
{'step': 1, 'epoch': 0, 'loss': 0.7590267062187195, 'lr': 1.05318588730911e-07, 'gradient_norm': 14.376425743103027, 'skipped': 1, 'elapsed_minutes': 0.8696492485666644}
{'step': 20, 'epoch': 0, 'loss': 0.8383651971817017, 'lr': 1.1058451816745656e-06, 'gradient_norm': 10.429871559143066, 'skipped': 1, 'elapsed_minutes': 0.9902932532000023}
{'step': 40, 'epoch': 0, 'loss': 0.593737006187439, 'lr': 2.1590310689836755e-06, 'gradient_norm': 6.693352222442627, 'skipped': 1, 'elapsed_minutes': 1.118144843216669}
AMP update skipped; retrying the same cases
{'step': 60, 'epoch': 0, 'loss': 0.5536811947822571, 'lr': 3.212216956292786e-06, 'gradient_norm': 7.491452217102051, 'skipped': 2, 'elapsed_minutes': 1.2514312436500026}
{'step': 80, 'epoch': 0, 'loss': 0.8125592470169067, 'lr': 4.265402843601897e-06, 'gradient_norm': 12.521574020385742, 'skipped': 2, 'elapsed_minutes': 1.37753257

Evaluate:   0%|          | 0/314 [00:00<?, ?it/s]

{'step': 3957, 'epoch': 1, 'validation_macro_f1': 0.6716194527189026}
{'step': 3960, 'epoch': 1, 'loss': 0.3352891206741333, 'lr': 9.893892570236598e-05, 'gradient_norm': 3.3225109577178955, 'skipped': 3, 'elapsed_minutes': 27.218436184216664}
{'step': 3980, 'epoch': 1, 'loss': 1.072679042816162, 'lr': 9.891831569511761e-05, 'gradient_norm': 4.760459899902344, 'skipped': 3, 'elapsed_minutes': 27.344310716316667}
{'step': 4000, 'epoch': 1, 'loss': 0.5975131392478943, 'lr': 9.889750988143368e-05, 'gradient_norm': 4.296170711517334, 'skipped': 3, 'elapsed_minutes': 27.470274950250005}
{'step': 4020, 'epoch': 1, 'loss': 0.4296521842479706, 'lr': 9.887650835407533e-05, 'gradient_norm': 2.6155002117156982, 'skipped': 3, 'elapsed_minutes': 27.613493090416668}
{'step': 4040, 'epoch': 1, 'loss': 0.3932949900627136, 'lr': 9.885531120667628e-05, 'gradient_norm': 2.200737237930298, 'skipped': 3, 'elapsed_minutes': 27.739562921583335}
{'step': 4060, 'epoch': 1, 'loss': 0.591460645198822, 'lr': 9.88

Evaluate:   0%|          | 0/314 [00:00<?, ?it/s]

... (9 outputs left out)

Evaluate:   0%|          | 0/314 [00:00<?, ?it/s]

{'step': 27699, 'epoch': 7, 'validation_macro_f1': 0.6818007851650651}
{'step': 27700, 'epoch': 7, 'loss': 0.0032178277615457773, 'lr': 1.3868069941313813e-05, 'gradient_norm': 0.5623546242713928, 'skipped': 17, 'elapsed_minutes': 184.98036178849998}
{'step': 27720, 'epoch': 7, 'loss': 0.002613452263176441, 'lr': 1.382962092856098e-05, 'gradient_norm': 0.22778016328811646, 'skipped': 17, 'elapsed_minutes': 185.1180024041833}
{'step': 27740, 'epoch': 7, 'loss': 0.0018267692066729069, 'lr': 1.3791355470779418e-05, 'gradient_norm': 0.0389779768884182, 'skipped': 17, 'elapsed_minutes': 185.24240236945}
{'step': 27760, 'epoch': 7, 'loss': 0.003481354797258973, 'lr': 1.375327373857273e-05, 'gradient_norm': 0.15965116024017334, 'skipped': 17, 'elapsed_minutes': 185.36679366886668}
{'step': 27780, 'epoch': 7, 'loss': 0.002224629744887352, 'lr': 1.3715375901725417e-05, 'gradient_norm': 0.0401020310819149, 'skipped': 17, 'elapsed_minutes': 185.49148281568336}
{'step': 27800, 'epoch': 7, 'loss': 

Evaluate:   0%|          | 0/314 [00:00<?, ?it/s]

{'step': 31656, 'epoch': 8, 'validation_macro_f1': 0.6768482198735974}
{'name': 'model1_lorma', 'epoch': 8, 'position': 0, 'step': 31656, 'best_macro_f1': 0.7076123004247933, 'bad_epochs': 3, 'complete': True, 'skipped': 19, 'stop_reason': 'epochs', 'parameters': {'total': 121630723, 'trainable': 1814147, 'adapted_modules': 64}, 'session_seconds': 12662.928957715001}
{'name': 'model1_lorma', 'epoch': 8, 'position': 0, 'step': 31656, 'best_macro_f1': 0.7076123004247933, 'bad_epochs': 3, 'complete': True, 'skipped': 19, 'stop_reason': 'epochs', 'parameters': {'total': 121630723, 'trainable': 1814147, 'adapted_modules': 64}, 'session_seconds': 12662.928957715001}


In [27]:
lorma_validation = validate_run(LORMA_RUN)
lorma_training_sample = training_sample_report(LORMA_RUN)

Evaluate:   0%|          | 0/314 [00:00<?, ?it/s]

Examples: 2511
Macro-F1: 0.707612
Precision: 0.736323
Recall: 0.652105

0 {'precision': 0.6862464183381088, 'recall': 0.7651757188498403, 'f1-score': 0.7235649546827795, 'support': 1252.0}
1 {'precision': 0.736322869955157, 'recall': 0.6521048451151708, 'f1-score': 0.6916596461668071, 'support': 1259.0}


Training sample:   0%|          | 0/16 [00:00<?, ?it/s]

Training sample Macro-F1: 0.9138047138047138 | cases: 128


In [28]:
lorma_submission = export_test(LORMA_RUN)
print("Submission:", lorma_submission)

Test cases:   0%|          | 0/190 [00:00<?, ?it/s]

/kaggle/working/phase4_cjpe/runs/model1_lorma/test/1d2745c73849/model1_lorma_submission.zip
Submission: /kaggle/working/phase4_cjpe/runs/model1_lorma/test/1d2745c73849/model1_lorma_submission.zip


### Model 2 — full fine-tuning

Artifacts: `phase4_cjpe/runs/model2_full`.


In [1]:
register('model2_full', 'model2', 'full', **DOCUMENT_POLICY)
coverage_report('model2', EXPERIMENTS['model2_full']['config'])
capacity_check('model2_full')


NameError: name 'register' is not defined

In [ ]:
model2_full_report = train_run('model2_full')
print(model2_full_report)


In [ ]:
model2_full_validation = validate_run('model2_full')
model2_full_training_sample = training_sample_report('model2_full')


In [ ]:
model2_full_submission = export_test('model2_full')


### Model 3 — AdaLoRA

Artifacts: `phase4_cjpe/runs/model3_adalora`.


In [ ]:
register('model3_adalora', 'model3', 'adalora', **DOCUMENT_POLICY)
coverage_report('model3', EXPERIMENTS['model3_adalora']['config'])
capacity_check('model3_adalora')


In [ ]:
model3_adalora_report = train_run('model3_adalora')
print(model3_adalora_report)


In [ ]:
model3_adalora_validation = validate_run('model3_adalora')
model3_adalora_training_sample = training_sample_report('model3_adalora')


In [ ]:
model3_adalora_submission = export_test('model3_adalora')


## 14. Model 4 distillation

Select a completed teacher using development Macro-F1. Its logits are cached on training cases only. The student retains alternate blocks from the Phase 1 CNN; the retained weights and dilations are copied exactly. The code checks that the student has fewer parameters than the teacher. Distillation uses label cross-entropy plus temperature-scaled KL divergence.

The following no-KD control uses the same reduced student and document policy. The optional full CNN reference measures the effect of removing blocks. Neither control replaces the required distillation experiment.


In [ ]:
TEACHER_NAME=choose_teacher()
STUDENT_KEEP=list(range(0,BASES['model4']['config']['layers'],2))
print('Retained Phase 1 CNN blocks:',STUDENT_KEEP)
teacher_logits,teacher_reference=cache_teacher(TEACHER_NAME)
print('Teacher:',teacher_reference)
del teacher_logits


### Model 4 — smaller distilled student

Artifacts: `phase4_cjpe/runs/model4_distilled`.


In [ ]:
register('model4_distilled', 'model4', 'kd', student_keep=STUDENT_KEEP, **DOCUMENT_POLICY)
coverage_report('model4', EXPERIMENTS['model4_distilled']['config'])
capacity_check('model4_distilled')


In [ ]:
model4_distilled_report = train_run('model4_distilled', teacher_name=TEACHER_NAME)
print(model4_distilled_report)


In [ ]:
model4_distilled_validation = validate_run('model4_distilled')
model4_distilled_training_sample = training_sample_report('model4_distilled')


In [ ]:
model4_distilled_submission = export_test('model4_distilled')


### Optional — smaller Model 4 without distillation

Artifacts: `phase4_cjpe/runs/model4_small_control`.


In [ ]:
register('model4_small_control', 'model4', 'control', student_keep=STUDENT_KEEP, **DOCUMENT_POLICY)
coverage_report('model4', EXPERIMENTS['model4_small_control']['config'])
capacity_check('model4_small_control')


In [ ]:
model4_small_control_report = train_run('model4_small_control')
print(model4_small_control_report)


In [ ]:
model4_small_control_validation = validate_run('model4_small_control')
model4_small_control_training_sample = training_sample_report('model4_small_control')


In [ ]:
model4_small_control_submission = export_test('model4_small_control')


### Optional — full-size Model 4 reference

Artifacts: `phase4_cjpe/runs/model4_full_reference`.


In [ ]:
register('model4_full_reference', 'model4', 'full', **DOCUMENT_POLICY)
coverage_report('model4', EXPERIMENTS['model4_full_reference']['config'])
capacity_check('model4_full_reference')


In [ ]:
model4_full_reference_report = train_run('model4_full_reference')
print(model4_full_reference_report)


In [ ]:
model4_full_reference_validation = validate_run('model4_full_reference')
model4_full_reference_training_sample = training_sample_report('model4_full_reference')


In [ ]:
model4_full_reference_submission = export_test('model4_full_reference')


## 15. Compare and collect

The planned four-model set is Model 1 LoRMA, Model 2 full fine-tuning, Model 3 AdaLoRA and Model 4 distillation. Additional baselines and controls are separate experiments. Keep the professor's current submission rules in view when selecting the final four. The table checks completion; it does not submit anything automatically.


In [ ]:
for row in completed_runs(): print(row)
required=['model1_lorma','model2_full','model3_adalora','model4_distilled']
status={r['name']:r for r in completed_runs()}
for name in required:
    print(name, 'ready' if name in status and status[name]['complete'] else 'unfinished')
write_json(OUTPUT_ROOT/'experiment_comparison.json',completed_runs())


### Optional threshold comparison

Set the run name below, then compare a small fixed threshold grid using development data. A selected threshold is not a new adaptation technique. Exporting with it creates a separate folder. Run this only for a shortlisted model; repeated selection on the same development set can overfit it.


In [ ]:
THRESHOLD_RUN = None
if THRESHOLD_RUN is not None:
    selected_threshold=threshold_comparison(THRESHOLD_RUN)
    threshold_submission=export_test(THRESHOLD_RUN,threshold=selected_threshold)


## Method references

[ILDC/CJPE](https://aclanthology.org/2021.acl-long.313/), [long legal document classification](https://aclanthology.org/2022.nllp-1.11/), [LoRMA](https://github.com/Exploration-Lab/LoRMA), [AdaLoRA](https://huggingface.co/docs/peft/package_reference/adalora), and [knowledge distillation](https://arxiv.org/abs/1503.02531).